In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_bronze = spark.table("fmcg.bronze.customers")
display(df_bronze)
df_bronze.printSchema()
print("Bronze row count:", df_bronze.count())

customer_id,customer_name,city,read_timestamp,filename,file_size
789201,FitFuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-09-21T11:41:28.110Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404


root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- filename: string (nullable = true)
 |-- file_size: long (nullable = true)

Bronze row count: 39


In [0]:
df_silver = df_bronze

In [0]:
#Check duplicates



display(
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
df_silver = df_silver.dropDuplicates(["customer_id"])
print("Silver row count:", df_silver.count())

Silver row count: 35


In [0]:
#Clean customer name
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)
#Clean city
df_silver = df_silver.withColumn(
    "city",
    F.trim(F.col("city"))
)


city_mapping = {
    "Bengaluruu": "Bengaluru",
    "Bengalore": "Bengaluru",
    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi",
    "NewDelhee": "New Delhi"
}

df_silver = df_silver.replace(
    city_mapping,
    subset=["city"]
)

In [0]:
#Verify the city cleaning
display(
    df_silver
    .groupBy("city")
    .count()
    .orderBy("city")
)


city,count
null,4
Bengaluru,10
Hyderabad,11
New Delhi,10


In [0]:
#Create the standardized customer field
df_silver = df_silver.withColumn(
    "customer",
    F.concat_ws(
        "-",
        F.col("customer_name"),
        F.col("city")
    )
)


display(
    df_silver.select(
        "customer_id",
        "customer_name",
        "city",
        "customer"
    )
)

customer_id,customer_name,city,customer
789201,FitFuel Market,Bengaluru,FitFuel Market-Bengaluru
789202,FitFuel Market,Hyderabad,FitFuel Market-Hyderabad
789203,FitFuel Market,New Delhi,FitFuel Market-New Delhi
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru
789303,Athlete's Choice Store,New Delhi,Athlete's Choice Store-New Delhi
789101,Endurance Foods,Bengaluru,Endurance Foods-Bengaluru
789102,Endurance Foods,Hyderabad,Endurance Foods-Hyderabad
789103,Endurance Foods,New Delhi,Endurance Foods-New Delhi
789121,HydroBoost Nutrition,Hyderabad,HydroBoost Nutrition-Hyderabad
789122,HydroBoost Nutrition,New Delhi,HydroBoost Nutrition-New Delhi


In [0]:
df_silver = (
    df_silver
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
789201,FitFuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789203,FitFuel Market,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-New Delhi,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789303,Athlete's Choice Store,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-New Delhi,India,Sports Bar,Acquisition
789101,Endurance Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-Bengaluru,India,Sports Bar,Acquisition
789102,Endurance Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789121,HydroBoost Nutrition,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition


In [0]:
df_silver.printSchema()
display(df_silver)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- filename: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- customer: string (nullable = false)
 |-- market: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- channel: string (nullable = false)



customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
789201,FitFuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789203,FitFuel Market,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-New Delhi,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789303,Athlete's Choice Store,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-New Delhi,India,Sports Bar,Acquisition
789101,Endurance Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-Bengaluru,India,Sports Bar,Acquisition
789102,Endurance Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789121,HydroBoost Nutrition,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition


In [0]:
duplicate_count = (
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate customer IDs:", duplicate_count)

Duplicate customer IDs: 0


In [0]:
display(
    df_silver.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df_silver.columns
    ])
)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
0,0,4,0,0,0,0,0,0,0


In [0]:
display(
    df_silver.select(
        "customer_id",
        "customer_name",
        "city",
        "customer",
        "market",
        "platform",
        "channel"
    )
)

customer_id,customer_name,city,customer,market,platform,channel
789201,FitFuel Market,Bengaluru,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789203,FitFuel Market,New Delhi,FitFuel Market-New Delhi,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789303,Athlete's Choice Store,New Delhi,Athlete's Choice Store-New Delhi,India,Sports Bar,Acquisition
789101,Endurance Foods,Bengaluru,Endurance Foods-Bengaluru,India,Sports Bar,Acquisition
789102,Endurance Foods,Hyderabad,Endurance Foods-Hyderabad,India,Sports Bar,Acquisition
789103,Endurance Foods,New Delhi,Endurance Foods-New Delhi,India,Sports Bar,Acquisition
789121,HydroBoost Nutrition,Hyderabad,HydroBoost Nutrition-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition


In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fmcg.silver.customers")
)

In [0]:
df_silver_check = spark.table("fmcg.silver.customers")

display(df_silver_check)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
789622,EliteAthlete Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,EliteAthlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,PowerSnack hub,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,PowerSnack hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,GamePlan Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,GamePlan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,FitFuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,ZenAthlete foods,null,2026-09-21T11:41:28.110Z,customers.csv,1404,ZenAthlete foods,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,ZenAthlete Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,ZenAthlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
#Read Silver
df_gold = spark.table("fmcg.silver.customers")
display(df_gold)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
789622,EliteAthlete Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,EliteAthlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,PowerSnack hub,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,PowerSnack hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,GamePlan Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,GamePlan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,FitFuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,ZenAthlete foods,null,2026-09-21T11:41:28.110Z,customers.csv,1404,ZenAthlete foods,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,ZenAthlete Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,ZenAthlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
#Select Business Columns
df_gold = df_gold.select(
    "customer_id",
    "customer_name",
    "city",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
df_gold.printSchema()
display(df_gold)
print("Gold rows:", df_gold.count())

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



customer_id,customer_name,city,customer,market,platform,channel
789622,EliteAthlete Nutrition,New Delhi,EliteAthlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,PowerSnack hub,Hyderabad,PowerSnack hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,GamePlan Foods,Bengaluru,GamePlan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,FitFuel Market,Bengaluru,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,ZenAthlete foods,null,ZenAthlete foods,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,ZenAthlete Foods,Hyderabad,ZenAthlete Foods-Hyderabad,India,Sports Bar,Acquisition


Gold rows: 35


In [0]:
#Write Gold Table
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.sb_dim_customers")
)

In [0]:
#VERIFY THE GOLD TABLE
df_gold_check = spark.table("fmcg.gold.sb_dim_customers")
display(df_gold_check)
print("Gold customer count:", df_gold_check.count())

customer_id,customer_name,city,customer,market,platform,channel
789622,EliteAthlete Nutrition,New Delhi,EliteAthlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,PowerSnack hub,Hyderabad,PowerSnack hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,GamePlan Foods,Bengaluru,GamePlan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,FitFuel Market,Bengaluru,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,ZenAthlete foods,null,ZenAthlete foods,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,ZenAthlete Foods,Hyderabad,ZenAthlete Foods-Hyderabad,India,Sports Bar,Acquisition


Gold customer count: 35


In [0]:
#Check the existing Gold schema
display(
    spark.sql("SHOW TABLES IN fmcg.gold")
)

database,tableName,isTemporary
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,sb_dim_customers,false


In [0]:
df_parent_customers = spark.table("fmcg.gold.dim_customers")

display(df_parent_customers)

customer_code,customer,market,platform,channel
70002017,Sporty Essentials,India,Brick & Mortar,Retailer
70002018,PlayMore Sports,India,Brick & Mortar,Retailer
90002001,SportsHub,India,E-Commerce,Retailer
90002002,PowerPlay Sports,India,Brick & Mortar,Retailer
90002003,Winning Gear,India,Brick & Mortar,Retailer
90002004,Atlikon Sports India,India,Brick & Mortar,Direct
90002005,Atlikon Official,India,E-Commerce,Direct
90002006,Atlikon Essentials,India,Brick & Mortar,Direct
90002007,SportsMart,India,Brick & Mortar,Retailer
90002008,FitnessWorld,India,E-Commerce,Retailer


In [0]:
df_parent_customers.printSchema()
print("Parent customer count:", df_parent_customers.count())

root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

Parent customer count: 18


In [0]:
print("PARENT COLUMNS:")
print(df_parent_customers.columns)

print("\nCHILD COLUMNS:")
print(spark.table("fmcg.gold.sb_dim_customers").columns)

PARENT COLUMNS:
['customer_code', 'customer', 'market', 'platform', 'channel']

CHILD COLUMNS:
['customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel']


In [0]:
df_parent_customers.printSchema()
print(df_parent_customers.columns)

root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

['customer_code', 'customer', 'market', 'platform', 'channel']


In [0]:
display(df_parent_customers.limit(20))

customer_code,customer,market,platform,channel
70002017,Sporty Essentials,India,Brick & Mortar,Retailer
70002018,PlayMore Sports,India,Brick & Mortar,Retailer
90002001,SportsHub,India,E-Commerce,Retailer
90002002,PowerPlay Sports,India,Brick & Mortar,Retailer
90002003,Winning Gear,India,Brick & Mortar,Retailer
90002004,Atlikon Sports India,India,Brick & Mortar,Direct
90002005,Atlikon Official,India,E-Commerce,Direct
90002006,Atlikon Essentials,India,Brick & Mortar,Direct
90002007,SportsMart,India,Brick & Mortar,Retailer
90002008,FitnessWorld,India,E-Commerce,Retailer


In [0]:
df_child_customers = spark.table("fmcg.gold.sb_dim_customers")

display(df_child_customers.limit(20))

customer_id,customer_name,city,customer,market,platform,channel
789622,EliteAthlete Nutrition,New Delhi,EliteAthlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,PowerSnack hub,Hyderabad,PowerSnack hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,GamePlan Foods,Bengaluru,GamePlan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,FitFuel Market,Bengaluru,FitFuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,ZenAthlete foods,null,ZenAthlete foods,India,Sports Bar,Acquisition
789202,FitFuel Market,Hyderabad,FitFuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,HydroBoost Nutrition,New Delhi,HydroBoost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,ZenAthlete Foods,Hyderabad,ZenAthlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
#Check whether child customers already exist in parent
display(
    df_child_customers.alias("c")
    .join(
        df_parent_customers.alias("p"),
        F.col("c.customer") == F.col("p.customer"),
        "left"
    )
    .select(
        F.col("c.customer_id"),
        F.col("c.customer"),
        F.col("p.customer_code"),
        F.col("p.customer").alias("parent_customer")
    )
)

customer_id,customer,customer_code,parent_customer
789622,EliteAthlete Nutrition-New Delhi,null,null
789321,PowerSnack hub-Hyderabad,null,null
789601,Recovery Lane-Bengaluru,null,null
789720,GamePlan Foods-Bengaluru,null,null
789201,FitFuel Market-Bengaluru,null,null
789301,Athlete's Choice Store-Bengaluru,null,null
789420,ZenAthlete foods,null,null
789202,FitFuel Market-Hyderabad,null,null
789122,HydroBoost Nutrition-New Delhi,null,null
789421,ZenAthlete Foods-Hyderabad,null,null


In [0]:
allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)
display(df_silver.select("city").distinct())

city
Bengaluru
Hyderabad
New Delhi
null


In [0]:
#Fix Customer Name Casing
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(
        F.col("customer_name").isNull(),
        None
    ).otherwise(
        F.initcap("customer_name")
    )
)
display(
    df_silver.select("customer_name").distinct()
)

customer_name
Fitfuel Market
Athlete's Choice Store
Endurance Foods
Hydroboost Nutrition
Macrobite Superfoods
Powersnack Hub
Sprintx Nutrition
Zenathlete Foods
Peak Performance Store
Primefuel Nutrition


In [0]:
customer_city_fix = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}
df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)
display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
#Apply the Business Fix 
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")
    )
    .drop("fixed_city")
)
display(
    df_silver.filter(F.col("city").isNull())
)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel


In [0]:
display(
    df_silver.select(
        "customer_id",
        "customer_name",
        "city"
    ).orderBy("customer_id")
)

customer_id,customer_name,city
789101,Endurance Foods,Bengaluru
789102,Endurance Foods,Hyderabad
789103,Endurance Foods,New Delhi
789121,Hydroboost Nutrition,Hyderabad
789122,Hydroboost Nutrition,New Delhi
789201,Fitfuel Market,Bengaluru
789202,Fitfuel Market,Hyderabad
789203,Fitfuel Market,New Delhi
789220,Macrobite Superfoods,Bengaluru
789221,Macrobite Superfoods,Hyderabad


In [0]:
#Convert Customer ID to String
df_silver = df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)
df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- filename: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- customer: string (nullable = false)
 |-- market: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- channel: string (nullable = false)



In [0]:
#Rebuild the Standardized Customer Attributes


df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws(
            "-",
            "customer_name",
            F.coalesce(
                F.col("city"),
                F.lit("Unknown")
            )
        )
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(
    df_silver.select(
        "customer_id",
        "customer_name",
        "city",
        "customer",
        "market",
        "platform",
        "channel"
    )
)

customer_id,customer_name,city,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
print("Silver row count:", df_silver.count())
print(
    "Duplicate customer IDs:",
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Silver row count: 35
Duplicate customer IDs: 0


In [0]:
#Overwrite Silver
(
    df_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.silver.customers")
)
display(
    spark.table("fmcg.silver.customers")
)

customer_id,customer_name,city,read_timestamp,filename,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,2026-09-21T11:41:28.110Z,customers.csv,1404,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,2026-09-21T11:41:28.110Z,customers.csv,1404,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,2026-09-21T11:41:28.110Z,customers.csv,1404,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
df_silver = spark.table("fmcg.silver.customers")

df_gold = df_silver.select(
    "customer_id",
    "customer_name",
    "city",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
(
    df_gold.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.sb_dim_customers")
)

In [0]:
df_gold_check = spark.table("fmcg.gold.sb_dim_customers")

In [0]:
print("Gold customer count:", df_gold_check.count())

Gold customer count: 35


In [0]:
display(
    df_gold_check.select(
        "customer_id",
        "customer_name",
        "city",
        "customer",
        "market",
        "platform",
        "channel"
    )
)

customer_id,customer_name,city,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
df_parent_customers = spark.table("fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers")
print("Parent customers:", df_parent_customers.count())
print("Child customers:", df_child_customers.count())

Parent customers: 18
Child customers: 35


In [0]:
print("PARENT:")
df_parent_customers.printSchema()

print("CHILD:")
df_child_customers.printSchema()

PARENT:
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

CHILD:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



In [0]:
df_child_merge = df_child_customers.select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)
display(df_child_merge)

customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
print("Parent:", df_parent_customers.count())
print("Child:", df_child_merge.count())
display(df_child_merge)

Parent: 18
Child: 35


customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


MERGE CHILD CUSTOMERS INTO PARENT

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "fmcg.gold.dim_customers"
)

In [0]:
(
    target.alias("target")
    .merge(
        df_child_merge.alias("source"),
        "target.customer_code = source.customer_code"
    )
    .whenMatchedUpdate(
        set={
            "customer": "source.customer",
            "market": "source.market",
            "platform": "source.platform",
            "channel": "source.channel"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_code": "source.customer_code",
            "customer": "source.customer",
            "market": "source.market",
            "platform": "source.platform",
            "channel": "source.channel"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_parent_after_merge = spark.table(
    "fmcg.gold.dim_customers"
)

print(
    "Customer count after merge:",
    df_parent_after_merge.count()
)
display(df_parent_after_merge)

Customer count after merge: 53


customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
display(
    df_parent_after_merge
    .filter(
        F.col("customer_code").isin(
            [row.customer_code for row in df_child_merge.select("customer_code").collect()]
        )
    )
)

customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
print(
    "Customer count after merge:",
    spark.table("fmcg.gold.dim_customers").count()
)

Customer count after merge: 53


In [0]:
display(
    spark.sql("SHOW TABLES IN fmcg.bronze")
)

database,tableName,isTemporary
bronze,customers,false


PRODUCT PIPELINE

In [0]:
# ============================================================
# STAGE 5 — PRODUCT PIPELINE
# STEP 5.1 — Read Product CSV from S3
# ============================================================

product_path = "s3://sportsbar-dp24124/products/products.csv"

df_products_raw = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(product_path)
)

display(df_products_raw)

product_name,product_id,category
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy


In [0]:
# Check schema

df_products_raw.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)



In [0]:
# Check number of records

print("Product rows:", df_products_raw.count())
print("Product columns:", len(df_products_raw.columns))
display(df_products_raw.limit(20))

Product rows: 20
Product columns: 3


product_name,product_id,category
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy


In [0]:
# ============================================================
# STAGE 5 — PRODUCT PIPELINE
# STEP 5.4 — Create Product Bronze DataFrame
# ============================================================

df_products_bronze = (
    df_products_raw
    .withColumn("read_timestamp", F.current_timestamp())
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

display(df_products_bronze)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-21T14:06:38.355Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-21T14:06:38.355Z,products.csv,1388


In [0]:
print("Bronze product rows:", df_products_bronze.count())
df_products_bronze.printSchema()
display(
    df_products_bronze.limit(20)
)

Bronze product rows: 20
root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-21T14:07:06.443Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-21T14:07:06.443Z,products.csv,1388


In [0]:
# ============================================================
# STEP 5.6 — Write Product Bronze Delta Table
# ============================================================

(
    df_products_bronze.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.bronze.products")
)

In [0]:
df_products_bronze_check = spark.table(
    "fmcg.bronze.products"
)
print(
    "Bronze product count:",
    df_products_bronze_check.count()
)
display(df_products_bronze_check)

Bronze product count: 20


product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-21T14:07:21.594Z,products.csv,1388


SILVER LAYER PRODUCT


In [0]:
# ============================================================
# STAGE 6 — PRODUCT SILVER TRANSFORMATION
# STEP 6.1 — Read Product Bronze
# ============================================================

df_product_silver = spark.table("fmcg.bronze.products")

display(df_product_silver)
df_product_silver.printSchema()

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-21T14:07:21.594Z,products.csv,1388


root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
# Check duplicate product IDs

display(
    df_product_silver
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

product_id,count
25891101,2
25891102,2


In [0]:
display(
    df_product_silver
    .filter(
        ~F.col("product_id").cast("string").rlike("^[0-9]+$")
    )
)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,healthy snacks,2026-09-21T14:07:21.594Z,products.csv,1388


In [0]:
df_product_silver = (
    df_product_silver
    .filter(
        F.col("product_id").cast("string").rlike("^[0-9]+$")
    )
)
print(
    "Rows after removing invalid product IDs:",
    df_product_silver.count()
)

Rows after removing invalid product IDs: 19


In [0]:
df_product_silver = df_product_silver.withColumn(
    "product_id",
    F.col("product_id").cast("string")
)
df_product_silver.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
df_product_silver = df_product_silver.withColumn(
    "product_name",
    F.trim(F.col("product_name"))
)

In [0]:
df_product_silver = df_product_silver.withColumn(
    "category",
    F.trim(F.col("category"))
)
display(
    df_product_silver
    .select("category")
    .distinct()
    .orderBy("category")
)

category
electrolyte mix
energy bars
granola & cereals
healthy snacks
protien bars
recovery dairy


In [0]:
df_product_silver = (
    df_product_silver
    .dropDuplicates(["product_id"])
)
print(
    "Product rows after deduplication:",
    df_product_silver.count()
)
display(
    df_product_silver
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

Product rows after deduplication: 17


product_id,count


In [0]:
df_product_silver = df_product_silver.withColumn(
    "product_code",
    F.sha2(
        F.col("product_name"),
        256
    )
)
display(
    df_product_silver.select(
        "product_id",
        "product_name",
        "category",
        "product_code"
    )
)

product_id,product_name,category,product_code
25891101,SportsBar Energy Bar Choco Fudge (60g),energy bars,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891102,SportsBar Energy Bar Choco Fudge (40g),energy bars,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
25891103,SportsBar Energy Bar Choco Fudge (25g),energy bars,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
25891201,SportsBar Protien Bar Peanut Crunch (45g),protien bars,ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21
25891202,SportsBar Protien Bar Peanut Crunch (55g),protien bars,96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f
25891203,SportsBar Protien Bar Peanut Crunch (65g),protien bars,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9
25891301,SportsBar Granola Crunch Honey Almond (400g),granola & cereals,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
25891302,SportsBar Granola Crunch Honey Almond (300g),granola & cereals,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
25891303,SportsBar Granola Crunch Honey Almond (200g),granola & cereals,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
25891401,SportsBar Greek Yogurt Pro Vanilla (200g),recovery dairy,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


In [0]:
display(
    df_product_silver.select(
        "product_id",
        "product_name",
        "category",
        "product_code"
    )
)
print(
    "Final Silver product count:",
    df_product_silver.count()
)

product_id,product_name,category,product_code
25891101,SportsBar Energy Bar Choco Fudge (60g),energy bars,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891102,SportsBar Energy Bar Choco Fudge (40g),energy bars,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
25891103,SportsBar Energy Bar Choco Fudge (25g),energy bars,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
25891201,SportsBar Protien Bar Peanut Crunch (45g),protien bars,ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21
25891202,SportsBar Protien Bar Peanut Crunch (55g),protien bars,96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f
25891203,SportsBar Protien Bar Peanut Crunch (65g),protien bars,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9
25891301,SportsBar Granola Crunch Honey Almond (400g),granola & cereals,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
25891302,SportsBar Granola Crunch Honey Almond (300g),granola & cereals,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
25891303,SportsBar Granola Crunch Honey Almond (200g),granola & cereals,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
25891401,SportsBar Greek Yogurt Pro Vanilla (200g),recovery dairy,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


Final Silver product count: 17


In [0]:
# ============================================================
# STEP 6.11 — Write Product Silver Delta Table
# ============================================================

(
    df_product_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.silver.products")
)
df_product_silver_check = spark.table(
    "fmcg.silver.products"
)
print(
    "Silver product count:",
    df_product_silver_check.count()
)

display(df_product_silver_check)

Silver product count: 17


product_name,product_id,category,read_timestamp,file_name,file_size,product_code
SportsBar Oats Cookie Bites ChocoChip (180g),25891503,healthy snacks,2026-09-21T14:07:21.594Z,products.csv,1388,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-21T14:07:21.594Z,products.csv,1388,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Greek Yogurt Pro Vanilla (80g),25891403,recovery dairy,2026-09-21T14:07:21.594Z,products.csv,1388,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-21T14:07:21.594Z,products.csv,1388,96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f
SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,electrolyte mix,2026-09-21T14:07:21.594Z,products.csv,1388,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),25891602,electrolyte mix,2026-09-21T14:07:21.594Z,products.csv,1388,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-21T14:07:21.594Z,products.csv,1388,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-21T14:07:21.594Z,products.csv,1388,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05


GOLD LAYER


In [0]:
# ============================================================
# STAGE 7 — PRODUCT GOLD
# STEP 7.1 — Create Product Gold DataFrame
# ============================================================

df_product_gold = df_product_silver_check.select(
    "product_code",
    "product_name",
    "category"
)
display(df_product_gold)

product_code,product_name,category
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,SportsBar Oats Cookie Bites ChocoChip (180g),healthy snacks
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,SportsBar Energy Bar Choco Fudge (40g),energy bars
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,SportsBar Granola Crunch Honey Almond (300g),granola & cereals
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,SportsBar Greek Yogurt Pro Vanilla (80g),recovery dairy
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,SportsBar Granola Crunch Honey Almond (400g),granola & cereals
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,SportsBar Protien Bar Peanut Crunch (55g),protien bars
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),electrolyte mix
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),electrolyte mix
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,SportsBar Granola Crunch Honey Almond (200g),granola & cereals
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,SportsBar Greek Yogurt Pro Vanilla (200g),recovery dairy


In [0]:
(
    df_product_gold.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.sb_dim_products")
)

In [0]:
df_product_gold_check = spark.table(
    "fmcg.gold.sb_dim_products"
)

print(
    "Gold product count:",
    df_product_gold_check.count()
)

Gold product count: 17


In [0]:
display(df_product_gold_check)

product_code,product_name,category
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,SportsBar Oats Cookie Bites ChocoChip (180g),healthy snacks
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,SportsBar Energy Bar Choco Fudge (40g),energy bars
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,SportsBar Granola Crunch Honey Almond (300g),granola & cereals
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,SportsBar Greek Yogurt Pro Vanilla (80g),recovery dairy
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,SportsBar Granola Crunch Honey Almond (400g),granola & cereals
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,SportsBar Protien Bar Peanut Crunch (55g),protien bars
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),electrolyte mix
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),electrolyte mix
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,SportsBar Granola Crunch Honey Almond (200g),granola & cereals
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,SportsBar Greek Yogurt Pro Vanilla (200g),recovery dairy


PRODUCT PARENT + CHILD INTEGRATION

In [0]:
# ============================================================
# STAGE 8 — PRODUCT PARENT + CHILD INTEGRATION
# STEP 8.1 — List Gold Tables
# ============================================================

display(
    spark.sql("SHOW TABLES IN fmcg.gold")
)
df_parent_products = spark.table("fmcg.gold.sb_dim_products")
df_parent_products.printSchema()
print(
    "Parent product count:",
    df_parent_products.count()
)

database,tableName,isTemporary
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,sb_dim_customers,false
gold,sb_dim_products,false


root
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)

Parent product count: 17


In [0]:
# ============================================================
# STEP 8.3 — Load Child Gold Products
# ============================================================

df_child_products = spark.table(
    "fmcg.gold.sb_dim_products"
)

display(df_child_products)
print("Child product count:", df_child_products.count())

product_code,product_name,category
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,SportsBar Oats Cookie Bites ChocoChip (180g),healthy snacks
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,SportsBar Energy Bar Choco Fudge (40g),energy bars
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,SportsBar Granola Crunch Honey Almond (300g),granola & cereals
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,SportsBar Greek Yogurt Pro Vanilla (80g),recovery dairy
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,SportsBar Granola Crunch Honey Almond (400g),granola & cereals
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,SportsBar Protien Bar Peanut Crunch (55g),protien bars
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),electrolyte mix
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),electrolyte mix
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,SportsBar Granola Crunch Honey Almond (200g),granola & cereals
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,SportsBar Greek Yogurt Pro Vanilla (200g),recovery dairy


Child product count: 17


In [0]:
display(
    df_child_products.alias("c")
    .join(
        df_parent_products.alias("p"),
        F.col("c.product_code") == F.col("p.product_code"),
        "left"
    )
    .select(
        F.col("c.product_code"),
        F.col("c.product_name"),
        F.col("p.product_name").alias("parent_product")
    )
)

product_code,product_name,parent_product
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,SportsBar Oats Cookie Bites ChocoChip (180g),SportsBar Oats Cookie Bites ChocoChip (180g)
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,SportsBar Energy Bar Choco Fudge (40g),SportsBar Energy Bar Choco Fudge (40g)
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,SportsBar Granola Crunch Honey Almond (300g),SportsBar Granola Crunch Honey Almond (300g)
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,SportsBar Greek Yogurt Pro Vanilla (80g),SportsBar Greek Yogurt Pro Vanilla (80g)
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,SportsBar Granola Crunch Honey Almond (400g),SportsBar Granola Crunch Honey Almond (400g)
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,SportsBar Protien Bar Peanut Crunch (55g),SportsBar Protien Bar Peanut Crunch (55g)
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),SportsBar Electrolyte Mix Lemon-Lime (30 Sachets)
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),SportsBar Electrolyte Mix Lemon-Lime (15 Sachets)
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,SportsBar Granola Crunch Honey Almond (200g),SportsBar Granola Crunch Honey Almond (200g)
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,SportsBar Greek Yogurt Pro Vanilla (200g),SportsBar Greek Yogurt Pro Vanilla (200g)


In [0]:
unmatched_products = (
    df_child_products.alias("c")
    .join(
        df_parent_products.alias("p"),
        F.col("c.product_code") == F.col("p.product_code"),
        "left_anti"
    )
    .count()
)

print("New child products:", unmatched_products)

New child products: 0


In [0]:
from delta.tables import DeltaTable

product_target = DeltaTable.forName(
    spark,
    "fmcg.gold.dim_products"
)

In [0]:
(
    product_target.alias("target")
    .merge(
        df_child_products.alias("source"),
        "target.product_code = source.product_code"
    )
    .whenMatchedUpdate(
        set={
            "product": "source.product_name",
            "category": "source.category"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_code": "source.product_code",
            "product": "source.product_name",
            "category": "source.category"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_parent_products_after = spark.table(
    "fmcg.gold.dim_products"
)

print(
    "Parent product count after merge:",
    df_parent_products_after.count()
)
display(df_parent_products_after)

Parent product count after merge: 414


product_code,division,category,product,variant
ARCHDDE20D,Archery,Arrows,PX Carbon Arrow Set,12 Pack
ARCH158F41,Archery,Finger Tab,LX Leather Finger Tab,Universal
ARCHAFF0E4,Archery,Bow,AX Precision Recurve Bow,26 lbs
ARCH6B94F7,Archery,Bow,AX Precision Recurve Bow,30 lbs
ARCH5D1FE7,Archery,Bow Stringer,BX Bow Stringing Tool,Standard
ARCH7B49A9,Archery,Arm Guard,NX Archery Arm Guard,Medium
ARCH497D34,Archery,Target,TX Foam Archery Target,80 cm
ARCHE71D79,Archery,Arm Guard,NX Archery Arm Guard,Large
ARCHDD8749,Archery,Target,TX Foam Archery Target,60 cm
BADMC045D4,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Head Heavy


In [0]:
df_parent_products_now = spark.table("fmcg.gold.dim_products")

print("Current parent product count:", df_parent_products_now.count())

Current parent product count: 414


In [0]:
display(
    df_parent_products_now
    .groupBy("product_code")
    .count()
    .filter(F.col("count") > 1)
)

product_code,count


In [0]:
print(
    "Child product count:",
    spark.table("fmcg.gold.sb_dim_products").count()
)
print(
    "Distinct child product codes:",
    spark.table("fmcg.gold.sb_dim_products")
    .select("product_code")
    .distinct()
    .count()
)

Child product count: 17
Distinct child product codes: 17


In [0]:
# ============================================================
# STEP 8.8 — Check Product MERGE History
# ============================================================

display(
    spark.sql("""
        DESCRIBE HISTORY fmcg.gold.dim_products
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-09-21T14:22:41.000Z,78677850784393,suryatummalapali@gmail.com,MERGE,"Map(predicate -> [""(product_code#23659 = product_code#23667)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2775125923794426),c3c52b43-0892-47c7-b2a2-dd46552e5c9f,0921-110433-9d47folg-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3056, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2794, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 17, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1725, numTargetRowsUpdated -> 0, numOutputRows -> 17, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 17, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1031)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-21T10:06:32.000Z,78677850784393,suryatummalapali@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> Created by the file upload UI, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpoint.writeStatsAsJson"":""false"",""io.unitycatalog.tableId"":""4232a3c3-2c01-454d-99ec-416650405dc7""}, statsOnLoad -> true)",null,null,01f1b5a4-1c10-1491-8276-905a4ae9a53a,null,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 397, numOutputBytes -> 9077)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
print(
    "Current parent products:",
    spark.table("fmcg.gold.dim_products").count()
)

Current parent products: 414


In [0]:
df_parent_products_now = spark.table(
    "fmcg.gold.dim_products"
)

df_child_products = spark.table(
    "fmcg.gold.sb_dim_products"
)

child_in_parent = (
    df_child_products.alias("c")
    .join(
        df_parent_products_now.alias("p"),
        F.col("c.product_code") == F.col("p.product_code"),
        "inner"
    )
    .count()
)

print("Child products found in parent:", child_in_parent)

Child products found in parent: 17


In [0]:
# ============================================================
# STAGE 9 — GROSS PRICE PIPELINE
# STEP 9.1 — Check Existing Bronze Tables
# ============================================================

display(
    spark.sql("SHOW TABLES IN fmcg.bronze")
)

database,tableName,isTemporary
bronze,customers,false
bronze,products,false


GROSS


In [0]:
# ============================================================
# STAGE 9 — GROSS PRICE PIPELINE
# STEP 9.3 — Read Gross Price CSV from S3
# ============================================================

gross_price_path = "s3://sportsbar-dp24124/gross_price/gross_price.csv"

df_gross_price_raw = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(gross_price_path)
)

display(df_gross_price_raw)

product_id,month,gross_price
25891101,2025/07/01,-84
25891101,01/08/2025,unknown
25891101,2025/09/01,84
25891101,2025-10-01,83
25891101,2025-11-01,83
88888888,2025-12-01,-83
25891102,2025-07-01,68
25891102,2025-08-01,68
25891102,2025-09-01,68
25891102,2025-10-01,69


In [0]:
df_gross_price_raw.printSchema()
print("Gross price rows:", df_gross_price_raw.count())
print("Columns:", df_gross_price_raw.columns)

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)

Gross price rows: 108
Columns: ['product_id', 'month', 'gross_price']


In [0]:
# ============================================================
# STEP 9.4 — Create Gross Price Bronze DataFrame
# ============================================================

df_gross_price_bronze = (
    df_gross_price_raw
    .withColumn("read_timestamp", F.current_timestamp())
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

display(df_gross_price_bronze)

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-09-21T14:36:59.204Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-09-21T14:36:59.204Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-09-21T14:36:59.204Z,gross_price.csv,2741


In [0]:
# ============================================================
# STEP 9.6 — Write Gross Price Bronze
# ============================================================

(
    df_gross_price_bronze.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.bronze.gross_price")
)

In [0]:
df_gross_price_bronze_check = spark.table(
    "fmcg.bronze.gross_price"
)

print(
    "Bronze gross price count:",
    df_gross_price_bronze_check.count()
)

Bronze gross price count: 108


In [0]:
# ============================================================
# STAGE 10 — GROSS PRICE SILVER
# STEP 10.1 — Read Bronze
# ============================================================

df_gross_price_silver = spark.table(
    "fmcg.bronze.gross_price"
)

display(df_gross_price_silver)

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-09-21T14:37:12.019Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-09-21T14:37:12.019Z,gross_price.csv,2741


In [0]:
display(
    df_gross_price_silver.select(
        "product_id",
        "month",
        "gross_price"
    )
)

product_id,month,gross_price
25891101,2025/07/01,-84
25891101,01/08/2025,unknown
25891101,2025/09/01,84
25891101,2025-10-01,83
25891101,2025-11-01,83
88888888,2025-12-01,-83
25891102,2025-07-01,68
25891102,2025-08-01,68
25891102,2025-09-01,68
25891102,2025-10-01,69


In [0]:
display(
    df_gross_price_silver.filter(
        F.col("gross_price").isNull()
    )
)

product_id,month,gross_price,read_timestamp,file_name,file_size


In [0]:
df_gross_price_silver = df_gross_price_silver.withColumn(
    "gross_price",
    F.expr("try_cast(regexp_replace(gross_price, ',', '') AS DOUBLE)")
)

In [0]:
df_gross_price_silver.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



SILVER GROSS


In [0]:
# ============================================================
# STEP 10.5 — Inspect Month Values
# ============================================================

display(
    df_gross_price_silver
    .select("month")
    .distinct()
    .orderBy("month")
)

month
01-08-2025
01-09-2025
01-12-2025
01/08/2025
01/09/2025
01/10/2025
01/12/2025
2025-07-01
2025-08-01
2025-09-01


In [0]:
display(
    df_gross_price_silver
    .select("product_id")
    .distinct()
    .orderBy("product_id")
)

product_id
25891101
25891102
25891103
25891201
25891202
25891203
25891301
25891302
25891303
25891401


In [0]:
# ============================================================
# STEP 10.7 — Standardize Month to Date
# ============================================================

df_gross_price_silver = df_gross_price_silver.withColumn(
    "month_date",
    F.coalesce(
        F.try_to_date("month", "dd-MM-yyyy"),
        F.try_to_date("month", "dd/MM/yyyy"),
        F.try_to_date("month", "yyyy-MM-dd"),
        F.try_to_date("month", "yyyy/MM/dd")
    )
)

In [0]:
display(
    df_gross_price_silver.select(
        "month",
        "month_date"
    ).distinct().orderBy("month_date")
)

month,month_date
2025/07/01,2025-07-01
2025-07-01,2025-07-01
01/08/2025,2025-08-01
2025/08/01,2025-08-01
2025-08-01,2025-08-01
01-08-2025,2025-08-01
01/09/2025,2025-09-01
01-09-2025,2025-09-01
2025/09/01,2025-09-01
2025-09-01,2025-09-01


In [0]:
print(
    "Invalid dates:",
    df_gross_price_silver
    .filter(F.col("month_date").isNull())
    .count()
)



Invalid dates: 0


In [0]:
df_gross_price_silver = (
    df_gross_price_silver
    .withColumn("year", F.year("month_date"))
    .withColumn("month_number", F.month("month_date"))
)

In [0]:
# ============================================================
# STEP 10.12 — Join Gross Price with Product Silver
# ============================================================

df_products_silver = spark.table("fmcg.silver.products")

df_gross_price_silver = (
    df_gross_price_silver.alias("gp")
    .join(
        df_products_silver.select(
            "product_id",
            "product_code"
        ).alias("p"),
        F.col("gp.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("gp.*"),
        F.col("p.product_code")
    )
)

In [0]:
# ============================================================
# STEP 10.3 — Safely Convert Gross Price to DOUBLE
# ============================================================

df_gross_price_silver = df_gross_price_silver.withColumn(
    "gross_price",
    F.expr("try_cast(gross_price AS DOUBLE)")
)

In [0]:
display(
    df_gross_price_silver
    .filter(F.col("gross_price").isNull())
)
print(
    "Invalid/NULL gross prices:",
    df_gross_price_silver
    .filter(F.col("gross_price").isNull())
    .count()
)

product_id,month,gross_price,read_timestamp,file_name,file_size,month_date,year,month_number,product_code


Invalid/NULL gross prices: 0


In [0]:
display(
    spark.table("fmcg.bronze.gross_price")
    .filter(
        ~F.col("gross_price").rlike("^[0-9]+([.][0-9]+)?$")
    )
    .select(
        "product_id",
        "month",
        "gross_price"
    )
)

product_id,month,gross_price
25891101,2025/07/01,-84
25891101,01/08/2025,unknown
88888888,2025-12-01,-83
25891103,2025-10-01,not_available
25891202,2025-11-01,unknown
25891301,2025-12-01,-493
25891302,01/08/2025,unknown
25891302,2025-09-01,-275
25891302,2025-11-01,-300
25891402,2025-12-01,not_available


In [0]:
# ============================================================
# STEP 10.6 — Identify Invalid Gross Prices
# ============================================================

display(
    spark.table("fmcg.bronze.gross_price")
    .filter(
        (F.expr("try_cast(gross_price AS DOUBLE)").isNull()) |
        (F.expr("try_cast(gross_price AS DOUBLE)") <= 0)
    )
    .select(
        "product_id",
        "month",
        "gross_price"
    )
    .orderBy("product_id", "month")
)

product_id,month,gross_price
25891101,01/08/2025,unknown
25891101,2025/07/01,-84
25891103,2025-10-01,not_available
25891202,2025-11-01,unknown
25891301,2025-12-01,-493
25891302,01/08/2025,unknown
25891302,2025-09-01,-275
25891302,2025-11-01,-300
25891402,2025-12-01,not_available
25891403,2025-09-01,-49


In [0]:
invalid_price_count = (
    spark.table("fmcg.bronze.gross_price")
    .filter(
        (F.expr("try_cast(gross_price AS DOUBLE)").isNull()) |
        (F.expr("try_cast(gross_price AS DOUBLE)") <= 0)
    )
    .count()
)

print("Invalid gross price records:", invalid_price_count)

Invalid gross price records: 17


In [0]:
# ============================================================
# STEP 10.8 — Remove Invalid Gross Prices
# ============================================================

df_gross_price_silver = (
    df_gross_price_silver
    .filter(
        F.col("gross_price").isNotNull() &
        (F.col("gross_price") > 0)
    )
)

print("Valid gross price records:", df_gross_price_silver.count())

Valid gross price records: 91


In [0]:
display(
    df_gross_price_silver.select(
        "product_id",
        "month",
        "month_date",
        "year",
        "month_number",
        "gross_price"
    ).orderBy("product_id", "month_date")
)

product_id,month,month_date,year,month_number,gross_price
25891101,2025/09/01,2025-09-01,2025,9,84.0
25891101,2025-10-01,2025-10-01,2025,10,83.0
25891101,2025-11-01,2025-11-01,2025,11,83.0
25891102,2025-07-01,2025-07-01,2025,7,68.0
25891102,2025-08-01,2025-08-01,2025,8,68.0
25891102,2025-09-01,2025-09-01,2025,9,68.0
25891102,2025-10-01,2025-10-01,2025,10,69.0
25891102,2025-11-01,2025-11-01,2025,11,69.0
25891102,2025-12-01,2025-12-01,2025,12,69.0
25891103,2025/07/01,2025-07-01,2025,7,85.0


In [0]:
# ============================================================
# STEP 10.10 — Add Product Code
# ============================================================

df_products_silver = spark.table("fmcg.silver.products")

df_gross_price_silver = (
    df_gross_price_silver.alias("gp")
    .join(
        df_products_silver
        .select("product_id", "product_code")
        .alias("p"),
        F.col("gp.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("gp.product_id").alias("product_id"),
        F.col("gp.month").alias("month"),
        F.col("gp.month_date").alias("month_date"),
        F.col("gp.year").alias("year"),
        F.col("gp.month_number").alias("month_number"),
        F.col("gp.gross_price").alias("gross_price"),
        F.col("gp.read_timestamp").alias("read_timestamp"),
        F.col("gp.file_name").alias("file_name"),
        F.col("gp.file_size").alias("file_size"),
        F.col("p.product_code").alias("product_code")
    )
)

In [0]:
display(
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .select(
        "product_id",
        "month",
        "gross_price"
    )
)

print(
    "Records without product_code:",
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .count()
)

product_id,month,gross_price
99999999,2025-10-01,108.0
88888888,2025-12-01,100.0
99999999,2025-09-01,132.0
88888888,2025-11-01,138.0
88888888,2025/10/01,300.0
99999999,2025/08/01,74.0
77777777,2025-07-01,49.0
25891502,2025-07-01,203.0
25891502,2025-08-01,203.0
99999999,2025-09-01,203.0


Records without product_code: 16


In [0]:
display(
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .select(
        "product_id",
        "month",
        "gross_price"
    )
    .orderBy("product_id", "month")
)

product_id,month,gross_price
25891502,01-12-2025,237.0
25891502,2025-07-01,203.0
25891502,2025-08-01,203.0
25891502,2025-11-01,237.0
25891502,2025/10/01,237.0
77777777,2025-07-01,49.0
88888888,2025-07-01,292.0
88888888,2025-10-01,296.0
88888888,2025-11-01,138.0
88888888,2025-11-01,187.0


In [0]:
print(
    "Distinct unmapped product IDs:",
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .select("product_id")
    .distinct()
    .count()
)

Distinct unmapped product IDs: 4


In [0]:
display(
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .groupBy("product_id")
    .agg(
        F.count("*").alias("unmapped_rows")
    )
    .orderBy("product_id")
)

product_id,unmapped_rows
25891502,5
77777777,1
88888888,6
99999999,4


In [0]:
unmapped_ids = (
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .select("product_id")
    .distinct()
)

display(
    unmapped_ids.alias("u")
    .join(
        spark.table("fmcg.bronze.products").alias("bp"),
        F.col("u.product_id").cast("string") ==
        F.col("bp.product_id").cast("string"),
        "left"
    )
    .select(
        F.col("u.product_id").alias("gross_price_product_id"),
        F.col("bp.product_id").alias("bronze_product_id"),
        F.col("bp.product_name"),
        F.col("bp.category")
    )
)

gross_price_product_id,bronze_product_id,product_name,category
77777777,null,null,null
99999999,null,null,null
25891502,null,null,null
88888888,null,null,null


In [0]:
# ============================================================
# STEP 10.14 — Remove Gross Price Records Without Product Code
# ============================================================

df_gross_price_silver = (
    df_gross_price_silver
    .filter(F.col("product_code").isNotNull())
)

print("Final Silver gross price records:", df_gross_price_silver.count())


Final Silver gross price records: 75


In [0]:
# ============================================================
# STEP 10.15 — Final Silver Validation
# ============================================================

print("Total rows:", df_gross_price_silver.count())

print(
    "NULL product codes:",
    df_gross_price_silver
    .filter(F.col("product_code").isNull())
    .count()
)

print(
    "NULL gross prices:",
    df_gross_price_silver
    .filter(F.col("gross_price").isNull())
    .count()
)

print(
    "Non-positive prices:",
    df_gross_price_silver
    .filter(F.col("gross_price") <= 0)
    .count()
)

print(
    "NULL dates:",
    df_gross_price_silver
    .filter(F.col("month_date").isNull())
    .count()
)

Total rows: 75
NULL product codes: 0
NULL gross prices: 0
Non-positive prices: 0
NULL dates: 0


In [0]:
# ============================================================
# STEP 10.16 — Final Silver Schema
# ============================================================

df_gross_price_silver = df_gross_price_silver.select(
    "product_id",
    "product_code",
    "month_date",
    "year",
    "month_number",
    "gross_price",
    "read_timestamp",
    "file_name",
    "file_size"
)

df_gross_price_silver.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- month_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month_number: integer (nullable = true)
 |-- gross_price: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
# ============================================================
# STEP 10.17 — Write Gross Price Silver Table
# ============================================================

(
    df_gross_price_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.silver.gross_price")
)

print("fmcg.silver.gross_price created successfully.")

fmcg.silver.gross_price created successfully.


In [0]:
# ============================================================
# STEP 10.18 — Verify Silver Table
# ============================================================

df_check = spark.table("fmcg.silver.gross_price")

print("Silver row count:", df_check.count())

display(df_check.orderBy("product_id", "month_date"))

Silver row count: 75


product_id,product_code,month_date,year,month_number,gross_price,read_timestamp,file_name,file_size
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-09-01,2025,9,84.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-10-01,2025,10,83.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-11-01,2025,11,83.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-07-01,2025,7,68.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-08-01,2025,8,68.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-09-01,2025,9,68.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-10-01,2025,10,69.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-11-01,2025,11,69.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-12-01,2025,12,69.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741
25891103,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-07-01,2025,7,85.0,2026-09-21T14:37:12.019Z,gross_price.csv,2741


In [0]:
# ============================================================
# STEP 11.1 — Inspect Gold Tables
# ============================================================

display(
    spark.sql("SHOW TABLES IN fmcg.gold")
)


database,tableName,isTemporary
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,sb_dim_customers,false
gold,sb_dim_products,false


In [0]:
# ============================================================
# STEP 11.2 — Inspect Gold Table Structures
# ============================================================

gold_tables = [
    row.tableName
    for row in spark.sql("SHOW TABLES IN fmcg.gold").collect()
]

print(gold_tables)
spark.table("fmcg.gold.dim_gross_price").printSchema()
display(
    spark.table("fmcg.gold.dim_gross_price").limit(10)
)

['dim_customers', 'dim_date', 'dim_gross_price', 'dim_products', 'fact_orders', 'sb_dim_customers', 'sb_dim_products']
root
 |-- product_code: string (nullable = true)
 |-- price_inr: long (nullable = true)
 |-- year: long (nullable = true)



product_code,price_inr,year
ARCHDDE20D,2750,2024
ARCH158F41,5740,2024
ARCHAFF0E4,4610,2024
ARCH6B94F7,3910,2024
ARCH5D1FE7,1610,2024
ARCH7B49A9,1610,2024
ARCH497D34,1100,2024
ARCHE71D79,5830,2024
ARCHDD8749,3930,2024
BADMC045D4,4500,2024


In [0]:
# ============================================================
# STEP 11.3 — Inspect Existing Gold Gross Price
# ============================================================

df_gold_gross_price = spark.table("fmcg.gold.dim_gross_price")

print("Existing Gold rows:", df_gold_gross_price.count())

display(
    df_gold_gross_price
    .orderBy("year", "product_code")
    .limit(20)
)

Existing Gold rows: 794


product_code,price_inr,year
ARCH158F41,5740,2024
ARCH497D34,1100,2024
ARCH5D1FE7,1610,2024
ARCH6B94F7,3910,2024
ARCH7B49A9,1610,2024
ARCHAFF0E4,4610,2024
ARCHDD8749,3930,2024
ARCHDDE20D,2750,2024
ARCHE71D79,5830,2024
BADM03A202,3980,2024


In [0]:
# ============================================================
# STEP 11.4 — Check Gold Duplicate Keys
# ============================================================

display(
    df_gold_gross_price
    .groupBy("product_code", "year")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

product_code,year,count


In [0]:
# ============================================================
# STEP 11.5 — Check Silver Product-Year Records
# ============================================================

display(
    df_gross_price_silver
    .groupBy("product_code", "year")
    .count()
    .orderBy(F.desc("count"))
)

product_code,year,count
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,2025,6
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025,6
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,2025,5
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,5
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,5
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,5
ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae,2025,5
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,5
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,2025,5
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,4


In [0]:
# ============================================================
# STEP 11.6 — Inspect Existing Gold Price Distribution
# ============================================================

display(
    df_gold_gross_price
    .groupBy("year")
    .agg(
        F.count("*").alias("products"),
        F.min("price_inr").alias("min_price"),
        F.max("price_inr").alias("max_price"),
        F.avg("price_inr").alias("avg_price")
    )
    .orderBy("year")
)

year,products,min_price,max_price,avg_price
2024,397,300,17920,4156.070528967254
2025,397,320,18960,4426.24685138539


In [0]:
# ============================================================
# STEP 11.7 — Confirm Gold Key
# ============================================================

print(
    "Duplicate product_code + year records:",
    df_gold_gross_price
    .groupBy("product_code", "year")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate product_code + year records: 0


In [0]:
# ============================================================
# STEP 11.8 — Compare Silver vs Existing Gold
# ============================================================

display(
    df_gross_price_silver
    .filter(F.col("year") == 2025)
    .join(
        df_gold_gross_price
        .filter(F.col("year") == 2025)
        .select(
            "product_code",
            F.col("price_inr").alias("existing_gold_price")
        ),
        "product_code",
        "left"
    )
    .select(
        "product_code",
        "month_date",
        "gross_price",
        "existing_gold_price"
    )
    .orderBy("product_code", "month_date")
)

product_code,month_date,gross_price,existing_gold_price
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-08-01,292.0,null
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-09-01,292.0,null
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-11-01,281.0,null
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-12-01,281.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-07-01,85.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-08-01,85.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-09-01,85.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-11-01,86.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-12-01,86.0,null
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025-07-01,456.0,null


In [0]:
# ============================================================
# STEP 11.9 — Calculate Annual Average
# ============================================================

df_gross_price_annual = (
    df_gross_price_silver
    .groupBy("product_code", "year")
    .agg(
        F.round(F.avg("gross_price"), 2).alias("price_inr")
    )
)

display(
    df_gross_price_annual
    .orderBy("product_code")
)

product_code,year,price_inr
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,286.5
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85.4
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,470.8
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,177.4
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442.0
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,293.67
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,49.75
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135.0
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,95.5
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,108.0


In [0]:
# ============================================================
# STEP 11.10 — Compare Calculated Annual Price vs Gold
# ============================================================

df_gold_compare = (
    df_gross_price_annual.alias("new")
    .join(
        df_gold_gross_price
        .filter(F.col("year") == 2025)
        .select(
            "product_code",
            F.col("price_inr").alias("existing_gold_price")
        )
        .alias("old"),
        "product_code",
        "left"
    )
    .select(
        "product_code",
        "year",
        F.col("new.price_inr").alias("calculated_price"),
        "existing_gold_price",
        F.round(
            F.col("new.price_inr") - F.col("existing_gold_price"),
            2
        ).alias("difference")
    )
    .orderBy("product_code")
)

display(df_gold_compare)

product_code,year,calculated_price,existing_gold_price,difference
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,286.5,null,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85.4,null,null
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,470.8,null,null
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,177.4,null,null
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442.0,null,null
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,293.67,null,null
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,49.75,null,null
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135.0,null,null
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,95.5,null,null
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,108.0,null,null


In [0]:
print(
    "Matching prices:",
    df_gold_compare
    .filter(F.abs(F.col("difference")) < 0.01)
    .count()
)

print(
    "Total products:",
    df_gold_compare.count()
)

Matching prices: 0
Total products: 17


In [0]:
# ============================================================
# STEP 11.11 — Inspect Existing vs Calculated Gold Prices
# ============================================================

display(
    df_gold_compare
    .select(
        "product_code",
        "year",
        "calculated_price",
        "existing_gold_price",
        "difference"
    )
    .orderBy("product_code")
)

product_code,year,calculated_price,existing_gold_price,difference
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,286.5,null,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85.4,null,null
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,470.8,null,null
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,177.4,null,null
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442.0,null,null
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,293.67,null,null
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,49.75,null,null
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135.0,null,null
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,95.5,null,null
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,108.0,null,null


In [0]:
# ============================================================
# STEP 11.12 — Compare Monthly Statistics
# ============================================================

display(
    df_gross_price_silver
    .groupBy("product_code", "year")
    .agg(
        F.count("*").alias("months_available"),
        F.round(F.min("gross_price"), 2).alias("min_monthly_price"),
        F.round(F.max("gross_price"), 2).alias("max_monthly_price"),
        F.round(F.avg("gross_price"), 2).alias("avg_monthly_price"),
        F.round(F.first("gross_price"), 2).alias("first_price"),
        F.round(F.last("gross_price"), 2).alias("last_price")
    )
    .join(
        df_gold_gross_price
        .filter(F.col("year") == 2025)
        .select("product_code", F.col("price_inr").alias("gold_price")),
        "product_code",
        "left"
    )
    .orderBy("product_code")
)

product_code,year,months_available,min_monthly_price,max_monthly_price,avg_monthly_price,first_price,last_price,gold_price
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,4,281.0,292.0,286.5,292.0,281.0,null
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,5,85.0,86.0,85.4,85.0,86.0,null
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,5,456.0,493.0,470.8,456.0,493.0,null
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,5,171.0,187.0,177.4,171.0,187.0,null
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,4,440.0,444.0,442.0,444.0,440.0,null
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,3,289.0,296.0,293.67,289.0,296.0,null
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,4,49.0,50.0,49.75,49.0,50.0,null
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,4,132.0,138.0,135.0,132.0,138.0,null
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,4,94.0,100.0,95.5,94.0,100.0,null
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,5,108.0,108.0,108.0,108.0,108.0,null


In [0]:
# ============================================================
# STEP 11.13 — Create Child Gold Gross Price
# ============================================================

df_child_gross_price = (
    df_gross_price_silver
    .groupBy("product_code", "year")
    .agg(
        F.round(F.avg("gross_price"), 0)
        .cast("long")
        .alias("price_inr")
    )
)

display(
    df_child_gross_price
    .orderBy("product_code")
)

product_code,year,price_inr
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,287
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,471
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,177
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,294
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,50
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,96
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,108


In [0]:
# ============================================================
# STEP 11.14 — Validate Child Gold
# ============================================================

print("Child Gold rows:", df_child_gross_price.count())

print(
    "Duplicate product_code + year:",
    df_child_gross_price
    .groupBy("product_code", "year")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "NULL prices:",
    df_child_gross_price
    .filter(F.col("price_inr").isNull())
    .count()
)

Child Gold rows: 17
Duplicate product_code + year: 0
NULL prices: 0


In [0]:
# ============================================================
# STEP 11.15 — Save Child Gold Gross Price
# ============================================================

(
    df_child_gross_price.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.sb_dim_gross_price")
)

print("fmcg.gold.sb_dim_gross_price created successfully.")
display(
    spark.table("fmcg.gold.sb_dim_gross_price")
)

fmcg.gold.sb_dim_gross_price created successfully.


product_code,year,price_inr
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,2025,77
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,294
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,287
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,2025,401
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,471
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,2025,288
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025,69


In [0]:
# ============================================================
# STEP 11.16 — Merge Gross Price into Parent Gold
# ============================================================

from delta.tables import DeltaTable

gross_price_target = DeltaTable.forName(
    spark,
    "fmcg.gold.dim_gross_price"
)

df_child_gross_price = spark.table(
    "fmcg.gold.sb_dim_gross_price"
)

In [0]:
(
    gross_price_target.alias("target")
    .merge(
        df_child_gross_price.alias("source"),
        """
        target.product_code = source.product_code
        AND target.year = source.year
        """
    )
    .whenMatchedUpdate(set={
        "price_inr": "source.price_inr"
    })
    .whenNotMatchedInsert(values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    })
    .execute()
)

print("Gross Price MERGE completed successfully.")

Gross Price MERGE completed successfully.


In [0]:
# ============================================================
# STEP 11.17 — Verify Parent Gold
# ============================================================

df_gold_gross_price_final = spark.table(
    "fmcg.gold.dim_gross_price"
)

print(
    "Final Gold gross price rows:",
    df_gold_gross_price_final.count()
)

Final Gold gross price rows: 811


In [0]:
display(
    df_gold_gross_price_final.alias("g")
    .join(
        df_child_gross_price.alias("c"),
        (
            (F.col("g.product_code") == F.col("c.product_code")) &
            (F.col("g.year") == F.col("c.year"))
        ),
        "inner"
    )
    .select(
        F.col("g.product_code"),
        F.col("g.year"),
        F.col("g.price_inr")
    )
    .orderBy("product_code")
)

product_code,year,price_inr
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025,287
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025,85
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025,471
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025,177
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025,442
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025,294
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025,50
91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,2025,135
96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,2025,96
ac8b5d1e6a188f56b5b78552548aa361603a6fffc1d3b026648e8006d335ec21,2025,108


In [0]:
print(
    "Child products present in Parent Gold:",
    df_gold_gross_price_final.alias("g")
    .join(
        df_child_gross_price.alias("c"),
        (
            (F.col("g.product_code") == F.col("c.product_code")) &
            (F.col("g.year") == F.col("c.year"))
        ),
        "inner"
    )
    .select("g.product_code", "g.year")
    .distinct()
    .count()
)

Child products present in Parent Gold: 17


In [0]:
display(
    spark.sql("SHOW TABLES IN fmcg.bronze")
)

database,tableName,isTemporary
bronze,customers,false
bronze,gross_price,false
bronze,products,false


In [0]:
from pyspark.sql import functions as F

In [0]:
# ============================================================
# STEP 12.1 — Read Orders Raw Data
# ============================================================

df_orders_raw = spark.read.csv(
    "s3://sportsbar-dp24124/orders/landing/",
    header=True,
    inferSchema=True
)

print("Orders rows:", df_orders_raw.count())

df_orders_raw.printSchema()

display(df_orders_raw.limit(10))

Orders rows: 51810
root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)



order_id,order_placement_date,customer_id,product_id,order_qty
FOCT62720602,"Tuesday, September 30, 2025",ABC987,25891301,71.0
FOCT62720602,"Tuesday, September 30, 2025",789720,25891502,125.0
FOCT62720602,"Tuesday, September 30, 2025",789720,25891403,462.0
FOCT62720602,"Tuesday, September 30, 2025",INVALID,25891601,133.0
FOCT62720602,"Tuesday, September 30, 2025",789720,25891602,79.0
FOCT62720602,2025/09/30,XYZ123,25891101,472.0
FOCT62720602,2025/09/30,ABC987,25891301,71.0
FOCT62402303,"Tuesday, September 30, 2025",789402,25891303,24.0
FOCT62402303,"Tuesday, September 30, 2025",789402,25891301,26.0
FOCT62402303,"Tuesday, September 30, 2025",789402,25891102,328.0


In [0]:
# ============================================================
# STEP 12.2 — Create Orders Bronze DataFrame
# ============================================================

df_orders_bronze = (
    df_orders_raw
    .withColumn("read_timestamp", F.current_timestamp())
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

df_orders_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
# ============================================================
# STEP 12.3 — Orders Data Quality Check
# ============================================================

print("Total orders:", df_orders_bronze.count())

print(
    "NULL order_id:",
    df_orders_bronze.filter(F.col("order_id").isNull()).count()
)

print(
    "NULL order_placement_date:",
    df_orders_bronze.filter(F.col("order_placement_date").isNull()).count()
)

print(
    "NULL customer_id:",
    df_orders_bronze.filter(F.col("customer_id").isNull()).count()
)

print(
    "NULL product_id:",
    df_orders_bronze.filter(F.col("product_id").isNull()).count()
)

print(
    "NULL order_qty:",
    df_orders_bronze.filter(F.col("order_qty").isNull()).count()
)

Total orders: 51810
NULL order_id: 0
NULL order_placement_date: 0
NULL customer_id: 0
NULL product_id: 0
NULL order_qty: 2607


In [0]:
# ============================================================
# STEP 12.4 — Check Duplicate Order IDs
# ============================================================

duplicate_orders = (
    df_orders_bronze
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate order IDs:",
    duplicate_orders.count()
)

display(duplicate_orders.orderBy(F.desc("count")).limit(20))

Duplicate order IDs: 12964


order_id,count
FJUL323221602,11
FNOV71203603,10
FJUL329303603,10
FNOV72221603,9
FAUG42201603,9
FNOV722301602,9
FSEP529303603,9
FSEP524221603,9
FNOV73501502,9
FNOV724702603,9


In [0]:
# ============================================================
# STEP 12.5 — Inspect Order Dates
# ============================================================

display(
    df_orders_bronze
    .select("order_placement_date")
    .distinct()
    .orderBy("order_placement_date")
    .limit(30)
)

order_placement_date
01-07-2025
01-08-2025
01-09-2025
01-10-2025
01-11-2025
01/07/2025
01/08/2025
01/09/2025
01/10/2025
01/11/2025


In [0]:
# ============================================================
# STEP 12.6 — Check Order Quantity
# ============================================================

print(
    "Zero quantity:",
    df_orders_bronze
    .filter(F.col("order_qty") == 0)
    .count()
)

print(
    "Negative quantity:",
    df_orders_bronze
    .filter(F.col("order_qty") < 0)
    .count()
)

display(
    df_orders_bronze
    .select("order_qty")
    .summary()
)

Zero quantity: 0
Negative quantity: 0


summary,order_qty
count,49203
mean,226.4467004044469
stddev,136.8109731257822
min,16.0
25%,106.0
50%,198.0
75%,346.0
max,500.0


In [0]:
# ============================================================
# STEP 12.7 — Write Orders Bronze Table
# ============================================================

(
    df_orders_bronze.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.bronze.orders")
)

print("fmcg.bronze.orders created successfully.")

fmcg.bronze.orders created successfully.


In [0]:
# ============================================================
# STEP 12.8 — Verify Orders Bronze
# ============================================================

df_orders_bronze_check = spark.table("fmcg.bronze.orders")

print(
    "Bronze Orders rows:",
    df_orders_bronze_check.count()
)

df_orders_bronze_check.printSchema()

display(
    df_orders_bronze_check.limit(10)
)

Bronze Orders rows: 51810
root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025/07/01,789320,25891203,150.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025/07/01,789320,25891301,46.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891403,null,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891201,354.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,"Tuesday, July 01, 2025",789320,25891501,249.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,01-07-2025,789320,25891301,46.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891302,40.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891502,133.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891503,145.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,"Tuesday, July 01, 2025",789401,25891203,429.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744


In [0]:
# ============================================================
# STEP 13.1 — Read Orders Bronze
# ============================================================

df_orders_silver = spark.table("fmcg.bronze.orders")

print("Bronze Orders rows:", df_orders_silver.count())

df_orders_silver.printSchema()

Bronze Orders rows: 51810
root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
# ============================================================
# STEP 13.2 — Convert Order Placement Date
# ============================================================

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

df_orders_silver = (
    df_orders_silver
    .withColumn(
        "order_date",
        F.coalesce(
            F.try_to_date(F.col("order_placement_date"), "EEEE, MMMM d, yyyy"),
            F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
            F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
            F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy")
        )
    )
)

display(
    df_orders_silver.select(
        "order_id",
        "order_placement_date",
        "order_date"
    ).limit(20)
)

order_id,order_placement_date,order_date
FJUL33320501,2025/07/01,2025-07-01
FJUL33320501,2025/07/01,2025-07-01
FJUL33320501,01-07-2025,2025-07-01
FJUL33320501,"Tuesday, July 01, 2025",2025-07-01
FJUL33320501,"Tuesday, July 01, 2025",2025-07-01
FJUL33320501,01-07-2025,2025-07-01
FJUL33401603,"Tuesday, July 01, 2025",2025-07-01
FJUL33401603,"Tuesday, July 01, 2025",2025-07-01
FJUL33401603,"Tuesday, July 01, 2025",2025-07-01
FJUL33401603,"Tuesday, July 01, 2025",2025-07-01


In [0]:
# ============================================================
# STEP 13.3 — Validate Order Dates
# ============================================================

print(
    "Invalid order dates:",
    df_orders_silver
    .filter(
        F.col("order_date").isNull() &
        F.col("order_placement_date").isNotNull()
    )
    .count()
)

print(
    "NULL order dates:",
    df_orders_silver
    .filter(F.col("order_date").isNull())
    .count()
)

Invalid order dates: 0
NULL order dates: 0


In [0]:
# ============================================================
# STEP 13.4 — Standardize Order IDs
# ============================================================

df_orders_silver = (
    df_orders_silver
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.col("product_id").cast("string"))
)
display(
    df_orders_silver.filter(
        (F.col("order_id") == "") |
        (F.col("customer_id") == "") |
        (F.col("product_id") == "")
    )
)

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,order_date


In [0]:
print(
    "NULL order_id:",
    df_orders_silver.filter(F.col("order_id").isNull()).count()
)

print(
    "NULL customer_id:",
    df_orders_silver.filter(F.col("customer_id").isNull()).count()
)

print(
    "NULL product_id:",
    df_orders_silver.filter(F.col("product_id").isNull()).count()
)

NULL order_id: 0
NULL customer_id: 0
NULL product_id: 0


In [0]:
# ============================================================
# STEP 13.5 — Duplicate Order Check
# ============================================================

duplicate_order_ids = (
    df_orders_silver
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate order IDs:",
    duplicate_order_ids.count()
)

display(
    duplicate_order_ids
    .orderBy(F.desc("count"))
    .limit(20)
)

Duplicate order IDs: 12964


order_id,count
FJUL323221602,11
FNOV71203603,10
FJUL329303603,10
FAUG42201603,9
FNOV724702603,9
FSEP524221603,9
FNOV725903603,9
FSEP530622603,9
FNOV722301602,9
FSEP529303603,9


In [0]:
# ============================================================
# STEP 13.6 — Validate Customer IDs
# ============================================================

df_customers_silver = spark.table("fmcg.silver.customers")

customer_check = (
    df_orders_silver
    .select("customer_id")
    .distinct()
    .alias("o")
    .join(
        df_customers_silver
        .select(
            F.col("customer_id").cast("string").alias("customer_id")
        )
        .distinct()
        .alias("c"),
        "customer_id",
        "left"
    )
)

print(
    "Unmapped customer IDs:",
    customer_check
    .filter(F.col("c.customer_id").isNull())
    .count()
)

Unmapped customer IDs: 3


In [0]:
# ============================================================
# STEP 13.7 — Validate Product IDs
# ============================================================

df_products_silver = spark.table("fmcg.silver.products")

product_check = (
    df_orders_silver
    .select("product_id")
    .distinct()
    .alias("o")
    .join(
        df_products_silver
        .select(
            F.col("product_id").cast("string").alias("product_id")
        )
        .distinct()
        .alias("p"),
        "product_id",
        "left"
    )
)

print(
    "Unmapped product IDs:",
    product_check
    .filter(F.col("p.product_id").isNull())
    .count()
)

Unmapped product IDs: 4


In [0]:
# ============================================================
# STEP 13.8 — Final Orders Silver DataFrame
# ============================================================

df_orders_silver = df_orders_silver.select(
    "order_id",
    "order_date",
    "customer_id",
    "product_id",
    "order_qty",
    "read_timestamp",
    "file_name",
    "file_size"
)

df_orders_silver.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
# ============================================================
# STEP 13.9 — Final Orders Silver Validation
# ============================================================

print("Total Silver orders:", df_orders_silver.count())

print(
    "NULL order dates:",
    df_orders_silver.filter(F.col("order_date").isNull()).count()
)

print(
    "NULL customer IDs:",
    df_orders_silver.filter(F.col("customer_id").isNull()).count()
)

print(
    "NULL product IDs:",
    df_orders_silver.filter(F.col("product_id").isNull()).count()
)

print(
    "Zero quantities:",
    df_orders_silver.filter(F.col("order_qty") == 0).count()
)

print(
    "Negative quantities:",
    df_orders_silver.filter(F.col("order_qty") < 0).count()
)

Total Silver orders: 51810
NULL order dates: 0
NULL customer IDs: 0
NULL product IDs: 0
Zero quantities: 0
Negative quantities: 0


In [0]:
# ============================================================
# STEP 13.10 — Write Orders Silver Table
# ============================================================

(
    df_orders_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.silver.orders")
)

print("fmcg.silver.orders created successfully.")

fmcg.silver.orders created successfully.


In [0]:
# ============================================================
# STEP 13.11 — Verify Orders Silver
# ============================================================

df_orders_silver_check = spark.table("fmcg.silver.orders")

print(
    "Orders Silver rows:",
    df_orders_silver_check.count()
)

df_orders_silver_check.printSchema()

display(
    df_orders_silver_check.limit(10)
)

Orders Silver rows: 51810
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



order_id,order_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025-07-01,789320,25891203,150.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891403,null,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891201,354.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891501,249.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891302,40.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891502,133.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891503,145.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891203,429.0,2026-09-21T16:40:58.878Z,orders_2025_07_01.csv,20744


In [0]:
# ============================================================
# STEP 14.1 — Inspect Parent Orders Gold
# ============================================================

df_gold_orders = spark.table("fmcg.gold.fact_orders")

print("Existing Gold Orders rows:", df_gold_orders.count())

df_gold_orders.printSchema()

Existing Gold Orders rows: 93055
root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: long (nullable = true)



In [0]:
# ============================================================
# STEP 14.2 — Check Orders Gold Grain
# ============================================================

duplicate_gold_orders = (
    df_gold_orders
    .groupBy(
        "date",
        "product_code",
        "customer_code"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate date + product + customer records:",
    duplicate_gold_orders.count()
)

display(
    duplicate_gold_orders
    .orderBy(F.desc("count"))
    .limit(20)
)

Duplicate date + product + customer records: 0


date,product_code,customer_code,count


In [0]:
# ============================================================
# STEP 14.3 — Check Silver Orders Grain
# ============================================================

duplicate_silver_orders = (
    df_orders_silver
    .groupBy(
        "order_date",
        "product_id",
        "customer_id"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Multiple orders for same date + product + customer:",
    duplicate_silver_orders.count()
)

display(
    duplicate_silver_orders
    .orderBy(F.desc("count"))
    .limit(20)
)

Multiple orders for same date + product + customer: 9127


order_date,product_id,customer_id,count
2025-10-01,25891402,ABC987,7
2025-10-29,25891201,XYZ123,6
2025-07-12,25891202,INVALID,6
2025-10-24,25891401,789121,5
2025-09-29,25891202,ABC987,5
2025-09-30,25891103,INVALID,5
2025-11-17,25891502,XYZ123,5
2025-08-13,25891201,INVALID,5
2025-11-13,25891603,789903,5
2025-09-02,25891502,ABC987,5


In [0]:
# ============================================================
# STEP 14.4 — Create Orders Child Gold
# ============================================================

df_child_orders = (
    df_orders_silver
    .groupBy(
        "order_date",
        "product_id",
        "customer_id"
    )
    .agg(
        F.sum("order_qty").cast("long").alias("sold_quantity")
    )
)

print("Child Gold rows:", df_child_orders.count())

display(
    df_child_orders
    .orderBy("order_date", "product_id", "customer_id")
    .limit(20)
)


Child Gold rows: 41126


order_date,product_id,customer_id,sold_quantity
2025-07-01,25891101,789101,823
2025-07-01,25891101,789122,390
2025-07-01,25891101,789202,822
2025-07-01,25891101,789203,683
2025-07-01,25891101,789321,345
2025-07-01,25891101,789401,454
2025-07-01,25891101,789403,379
2025-07-01,25891101,789420,800
2025-07-01,25891101,789503,362
2025-07-01,25891101,789520,770


In [0]:
# ============================================================
# STEP 14.5 — Map Product and Customer Codes
# ============================================================

df_products_silver = spark.table("fmcg.silver.products")
df_customers_silver = spark.table("fmcg.silver.customers")

df_child_orders = (
    df_child_orders.alias("o")
    .join(
        df_products_silver
        .select(
            F.col("product_id").cast("string").alias("product_id"),
            "product_code"
        )
        .alias("p"),
        F.col("o.product_id").cast("string") == F.col("p.product_id"),
        "left"
    )
    .join(
        df_customers_silver
        .select(
            F.col("customer_id").cast("string").alias("customer_id")
        )
        .alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("o.order_date").alias("date"),
        F.col("p.product_code").alias("product_code"),
        F.col("o.customer_id").alias("customer_code"),
        F.col("o.sold_quantity").alias("sold_quantity")
    )
)

In [0]:
# ============================================================
# STEP 14.6 — Validate Orders Child Gold
# ============================================================

print("Child Gold rows:", df_child_orders.count())

print(
    "NULL product codes:",
    df_child_orders
    .filter(F.col("product_code").isNull())
    .count()
)

print(
    "NULL customer codes:",
    df_child_orders
    .filter(F.col("customer_code").isNull())
    .count()
)

print(
    "NULL sold quantities:",
    df_child_orders
    .filter(F.col("sold_quantity").isNull())
    .count()
)

Child Gold rows: 41126
NULL product codes: 4443
NULL customer codes: 0
NULL sold quantities: 1722


In [0]:
# ============================================================
# STEP 14.7 — Identify Unmapped Order Products
# ============================================================

display(
    df_child_orders
    .filter(F.col("product_code").isNull())
    .groupBy("customer_code")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

customer_code,count
XYZ123,142
INVALID,140
ABC987,138
789420,130
789201,129
789301,128
789902,128
789203,126
789422,125
789702,123


In [0]:
display(
    df_orders_silver
    .select("product_id")
    .distinct()
    .alias("o")
    .join(
        df_products_silver
        .select(
            F.col("product_id").cast("string").alias("product_id")
        )
        .distinct()
        .alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .filter(F.col("p.product_id").isNull())
    .orderBy("o.product_id")
)

product_id,product_id
25891502,null
77777777,null
88888888,null
99999999,null


In [0]:
# ============================================================
# STEP 14.8 — Check NULL Order Quantities
# ============================================================

print(
    "NULL order_qty records:",
    df_orders_silver
    .filter(F.col("order_qty").isNull())
    .count()
)

display(
    df_orders_silver
    .filter(F.col("order_qty").isNull())
    .select(
        "order_id",
        "order_date",
        "customer_id",
        "product_id",
        "order_qty"
    )
    .limit(50)
)

NULL order_qty records: 2607


order_id,order_date,customer_id,product_id,order_qty
FJUL33320501,2025-07-01,789320,25891403,null
FJUL33401603,2025-07-01,789401,25891603,null
FJUL32202603,2025-07-01,789202,25891603,null
FJUL32903403,2025-07-01,789903,99999999,null
FJUL32903403,2025-07-01,789903,25891303,null
FJUL33501401,2025-07-01,789501,25891103,null
FJUL32103503,2025-07-01,789103,25891503,null
FJUL32702601,2025-07-01,789702,25891102,null
FJUL33403501,2025-07-01,789403,25891501,null
FJUL32420403,2025-07-01,789420,25891403,null


In [0]:
# ============================================================
# STEP 14.9 — Filter Invalid Orders Before Gold
# ============================================================

df_orders_gold_source = (
    df_orders_silver
    .filter(F.col("order_qty").isNotNull())
    .filter(F.col("order_qty") > 0)
    .filter(
        F.col("product_id").cast("string").isin(
            [
                row["product_id"]
                for row in df_products_silver
                .select("product_id")
                .distinct()
                .collect()
            ]
        )
    )
)

print("Clean source orders:", df_orders_gold_source.count())

Clean source orders: 44039


In [0]:
# ============================================================
# STEP 14.9 — Filter Invalid Orders Before Gold
# ============================================================

df_orders_gold_source = (
    df_orders_silver
    .filter(F.col("order_qty").isNotNull())
    .filter(F.col("order_qty") > 0)
    .join(
        df_products_silver
        .select(
            F.col("product_id").cast("string").alias("product_id")
        )
        .distinct(),
        "product_id",
        "left_semi"
    )
)

print("Clean source orders:", df_orders_gold_source.count())

Clean source orders: 44039


In [0]:
# ============================================================
# STEP 14.10 — Aggregate Orders
# ============================================================

df_child_orders = (
    df_orders_gold_source
    .groupBy(
        "order_date",
        "product_id",
        "customer_id"
    )
    .agg(
        F.sum("order_qty")
        .cast("long")
        .alias("sold_quantity")
    )
)

print("Aggregated Orders:", df_child_orders.count())

Aggregated Orders: 35125


In [0]:
# ============================================================
# STEP 14.11 — Map Product Code
# ============================================================

df_child_orders = (
    df_child_orders.alias("o")
    .join(
        df_products_silver
        .select(
            F.col("product_id").cast("string").alias("product_id"),
            "product_code"
        )
        .alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("o.order_date").alias("date"),
        F.col("p.product_code").alias("product_code"),
        F.col("o.customer_id").alias("customer_code"),
        F.col("o.sold_quantity").alias("sold_quantity")
    )
)

In [0]:
# ============================================================
# STEP 14.12 — Validate Orders Child Gold
# ============================================================

print("Child Gold rows:", df_child_orders.count())

print(
    "NULL product codes:",
    df_child_orders
    .filter(F.col("product_code").isNull())
    .count()
)

print(
    "NULL customer codes:",
    df_child_orders
    .filter(F.col("customer_code").isNull())
    .count()
)

print(
    "NULL sold quantities:",
    df_child_orders
    .filter(F.col("sold_quantity").isNull())
    .count()
)

print(
    "Zero sold quantities:",
    df_child_orders
    .filter(F.col("sold_quantity") == 0)
    .count()
)

Child Gold rows: 35125
NULL product codes: 0
NULL customer codes: 0
NULL sold quantities: 0
Zero sold quantities: 0


In [0]:
# ============================================================
# STEP 14.13 — Final Child Orders Gold
# ============================================================

df_child_orders = df_child_orders.select(
    "date",
    "product_code",
    "customer_code",
    "sold_quantity"
)

df_child_orders.printSchema()

print("Child Gold rows:", df_child_orders.count())

display(df_child_orders.limit(20))

root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: long (nullable = true)

Child Gold rows: 35125


date,product_code,customer_code,sold_quantity
2025-07-01,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,789320,497
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789101,58
2025-07-01,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789220,372
2025-07-01,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,ABC987,206
2025-07-01,96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,789403,472
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789702,29
2025-07-01,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789103,420
2025-07-01,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,789520,105
2025-07-01,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789902,860
2025-07-01,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,789202,45


In [0]:
# ============================================================
# STEP 14.14 — Validate Gold Grain
# ============================================================

duplicate_child_orders = (
    df_child_orders
    .groupBy(
        "date",
        "product_code",
        "customer_code"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate Gold keys:",
    duplicate_child_orders.count()
)

Duplicate Gold keys: 0


In [0]:
# ============================================================
# STEP 14.15 — Save Child Orders Gold
# ============================================================

(
    df_child_orders.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.sb_fact_orders")
)

print("fmcg.gold.sb_fact_orders created successfully.")
display(
    spark.table("fmcg.gold.sb_fact_orders").limit(20)
)

fmcg.gold.sb_fact_orders created successfully.


date,product_code,customer_code,sold_quantity
2025-07-01,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,789320,497
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789101,58
2025-07-01,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789220,372
2025-07-01,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,ABC987,206
2025-07-01,96b71be076213a8eca8a49bb2882401c1d8bd63d72c7a0a19f95a6b91fd8560f,789403,472
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789702,29
2025-07-01,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789103,420
2025-07-01,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,789520,105
2025-07-01,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789902,860
2025-07-01,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,789202,45


In [0]:
# ============================================================
# STEP 14.16 — Merge Orders into Parent Gold
# ============================================================

from delta.tables import DeltaTable

orders_target = DeltaTable.forName(
    spark,
    "fmcg.gold.fact_orders"
)

df_child_orders = spark.table(
    "fmcg.gold.sb_fact_orders"
)

(
    orders_target.alias("target")
    .merge(
        df_child_orders.alias("source"),
        """
        target.date = source.date
        AND target.product_code = source.product_code
        AND target.customer_code = source.customer_code
        """
    )
    .whenMatchedUpdate(set={
        "sold_quantity": "source.sold_quantity"
    })
    .whenNotMatchedInsert(values={
        "date": "source.date",
        "product_code": "source.product_code",
        "customer_code": "source.customer_code",
        "sold_quantity": "source.sold_quantity"
    })
    .execute()
)

print("Orders MERGE completed successfully.")

Orders MERGE completed successfully.


In [0]:
# ============================================================
# STEP 14.17 — Verify Parent Orders Gold
# ============================================================

df_gold_orders_final = spark.table(
    "fmcg.gold.fact_orders"
)

print(
    "Final Parent Gold Orders:",
    df_gold_orders_final.count()
)

Final Parent Gold Orders: 128180


In [0]:
# ============================================================
# STEP 14.18 — Verify Child Records in Parent
# ============================================================

child_in_parent = (
    df_gold_orders_final.alias("g")
    .join(
        df_child_orders.alias("c"),
        (
            (F.col("g.date") == F.col("c.date")) &
            (F.col("g.product_code") == F.col("c.product_code")) &
            (F.col("g.customer_code") == F.col("c.customer_code"))
        ),
        "inner"
    )
    .select(
        F.col("g.date"),
        F.col("g.product_code"),
        F.col("g.customer_code"),
        F.col("g.sold_quantity")
    )
    .distinct()
)

print(
    "Child records present in Parent:",
    child_in_parent.count()
)

Child records present in Parent: 35125


In [0]:
# ============================================================
# STEP 14.19 — Final Orders Gold Validation
# ============================================================

print(
    "Parent Gold rows:",
    df_gold_orders_final.count()
)

print(
    "NULL dates:",
    df_gold_orders_final
    .filter(F.col("date").isNull())
    .count()
)

print(
    "NULL product codes:",
    df_gold_orders_final
    .filter(F.col("product_code").isNull())
    .count()
)

print(
    "NULL customer codes:",
    df_gold_orders_final
    .filter(F.col("customer_code").isNull())
    .count()
)

print(
    "NULL sold quantities:",
    df_gold_orders_final
    .filter(F.col("sold_quantity").isNull())
    .count()
)

print(
    "Negative sold quantities:",
    df_gold_orders_final
    .filter(F.col("sold_quantity") < 0)
    .count()
)

Parent Gold rows: 128180
NULL dates: 0
NULL product codes: 0
NULL customer codes: 0
NULL sold quantities: 0
Negative sold quantities: 0


In [0]:
# ============================================================
# STEP 14.20 — Check Final Gold Grain
# ============================================================

print(
    "Duplicate Gold keys:",
    df_gold_orders_final
    .groupBy(
        "date",
        "product_code",
        "customer_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate Gold keys: 0


In [0]:
# ============================================================
# STEP 15.1 — Inspect Final Gold Layer
# ============================================================

display(
    spark.sql("SHOW TABLES IN fmcg.gold")
)

database,tableName,isTemporary
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,sb_dim_customers,false
gold,sb_dim_gross_price,false
gold,sb_dim_products,false
gold,sb_fact_orders,false


In [0]:
# ============================================================
# STEP 15.2 — Final Gold Row Counts
# ============================================================

for table_name in [
    "dim_customers",
    "dim_products",
    "dim_gross_price",
    "fact_orders"
]:
    count = spark.table(f"fmcg.gold.{table_name}").count()
    print(f"{table_name}: {count:,} rows")

dim_customers: 53 rows
dim_products: 414 rows
dim_gross_price: 811 rows
fact_orders: 128,180 rows


Load All Gold Tables


In [0]:
# ============================================================
# STEP 15.3 — Load Gold Tables
# ============================================================

df_fact_orders = spark.table("fmcg.gold.fact_orders")

df_dim_customers = spark.table("fmcg.gold.dim_customers")

df_dim_products = spark.table("fmcg.gold.dim_products")

df_dim_gross_price = spark.table("fmcg.gold.dim_gross_price")

print("Orders:", df_fact_orders.count())
print("Customers:", df_dim_customers.count())
print("Products:", df_dim_products.count())
print("Gross Price:", df_dim_gross_price.count())

Orders: 128180
Customers: 53
Products: 414
Gross Price: 811


In [0]:
# ============================================================
# STEP 15.4 — Add Order Year
# ============================================================

df_sales = (
    df_fact_orders
    .withColumn("year", F.year("date"))
)

display(
    df_sales.limit(10)
)

date,product_code,customer_code,sold_quantity,year
2024-01-01,ARCHDDE20D,70002017,161,2024
2024-01-01,ARCH158F41,70002017,133,2024
2024-01-01,ARCHAFF0E4,70002017,76,2024
2024-01-01,ARCH6B94F7,70002017,92,2024
2024-01-01,ARCH5D1FE7,70002017,117,2024
2024-01-01,ARCH7B49A9,70002017,36,2024
2024-01-01,ARCH497D34,70002017,98,2024
2024-01-01,ARCHE71D79,70002017,156,2024
2024-01-01,BADM88C384,70002017,28,2024
2024-01-01,BADMA5EBA3,70002017,33,2024


In [0]:
# ============================================================
# STEP 15.5 — Join Gross Price
# ============================================================

df_sales = (
    df_sales.alias("o")
    .join(
        df_dim_gross_price.alias("gp"),
        (
            (F.col("o.product_code") == F.col("gp.product_code")) &
            (F.col("o.year") == F.col("gp.year"))
        ),
        "left"
    )
    .select(
        F.col("o.date"),
        F.col("o.year"),
        F.col("o.product_code"),
        F.col("o.customer_code"),
        F.col("o.sold_quantity"),
        F.col("gp.price_inr")
    )
)

display(df_sales.limit(10))

date,year,product_code,customer_code,sold_quantity,price_inr
2024-01-01,2024,ARCHDDE20D,70002017,161,2750
2024-01-01,2024,ARCH158F41,70002017,133,5740
2024-01-01,2024,ARCHAFF0E4,70002017,76,4610
2024-01-01,2024,ARCH6B94F7,70002017,92,3910
2024-01-01,2024,ARCH5D1FE7,70002017,117,1610
2024-01-01,2024,ARCH7B49A9,70002017,36,1610
2024-01-01,2024,ARCH497D34,70002017,98,1100
2024-01-01,2024,ARCHE71D79,70002017,156,5830
2024-01-01,2024,BADM88C384,70002017,28,790
2024-01-01,2024,BADMA5EBA3,70002017,33,5910


In [0]:
# ============================================================
# STEP 15.6 — Validate Gross Price Mapping
# ============================================================

print(
    "Orders without gross price:",
    df_sales
    .filter(F.col("price_inr").isNull())
    .count()
)

print(
    "Orders with gross price:",
    df_sales
    .filter(F.col("price_inr").isNotNull())
    .count()
)

Orders without gross price: 0
Orders with gross price: 128180


In [0]:
# ============================================================
# STEP 15.7 — Calculate Revenue
# ============================================================

df_sales = (
    df_sales
    .withColumn(
        "revenue_inr",
        F.col("sold_quantity") * F.col("price_inr")
    )
)

display(
    df_sales.limit(20)
)

date,year,product_code,customer_code,sold_quantity,price_inr,revenue_inr
2024-01-01,2024,ARCHDDE20D,70002017,161,2750,442750
2024-01-01,2024,ARCH158F41,70002017,133,5740,763420
2024-01-01,2024,ARCHAFF0E4,70002017,76,4610,350360
2024-01-01,2024,ARCH6B94F7,70002017,92,3910,359720
2024-01-01,2024,ARCH5D1FE7,70002017,117,1610,188370
2024-01-01,2024,ARCH7B49A9,70002017,36,1610,57960
2024-01-01,2024,ARCH497D34,70002017,98,1100,107800
2024-01-01,2024,ARCHE71D79,70002017,156,5830,909480
2024-01-01,2024,BADM88C384,70002017,28,790,22120
2024-01-01,2024,BADMA5EBA3,70002017,33,5910,195030


In [0]:
# ============================================================
# STEP 15.8 — Validate Revenue
# ============================================================

print(
    "NULL revenue:",
    df_sales
    .filter(F.col("revenue_inr").isNull())
    .count()
)

print(
    "Negative revenue:",
    df_sales
    .filter(F.col("revenue_inr") < 0)
    .count()
)

print(
    "Total revenue INR:",
    df_sales
    .agg(F.sum("revenue_inr"))
    .collect()[0][0]
)

NULL revenue: 0
Negative revenue: 0
Total revenue INR: 105367756383


In [0]:
# ============================================================
# STEP 15.9 — Add Product Attributes
# ============================================================

df_products_lookup = (
    df_dim_products
    .select(
        F.col("product_code").alias("lookup_product_code"),
        F.col("product"),
        F.col("category")
    )
)

df_sales = (
    df_sales
    .join(
        df_products_lookup,
        df_sales["product_code"] == df_products_lookup["lookup_product_code"],
        "left"
    )
    .drop("lookup_product_code")
)

display(
    df_sales.limit(20)
)

date,year,product_code,customer_code,sold_quantity,price_inr,revenue_inr,product,category
2024-01-01,2024,ARCHDDE20D,70002017,161,2750,442750,PX Carbon Arrow Set,Arrows
2024-01-01,2024,ARCH158F41,70002017,133,5740,763420,LX Leather Finger Tab,Finger Tab
2024-01-01,2024,ARCHAFF0E4,70002017,76,4610,350360,AX Precision Recurve Bow,Bow
2024-01-01,2024,ARCH6B94F7,70002017,92,3910,359720,AX Precision Recurve Bow,Bow
2024-01-01,2024,ARCH5D1FE7,70002017,117,1610,188370,BX Bow Stringing Tool,Bow Stringer
2024-01-01,2024,ARCH7B49A9,70002017,36,1610,57960,NX Archery Arm Guard,Arm Guard
2024-01-01,2024,ARCH497D34,70002017,98,1100,107800,TX Foam Archery Target,Target
2024-01-01,2024,ARCHE71D79,70002017,156,5830,909480,NX Archery Arm Guard,Arm Guard
2024-01-01,2024,BADM88C384,70002017,28,790,22120,AX PowerDrive Badminton Racket,Badminton Racket
2024-01-01,2024,BADMA5EBA3,70002017,33,5910,195030,BX AeroLite Badminton Racket,Badminton Racket


In [0]:
# ============================================================
# STEP 15.10 — Validate Product Mapping
# ============================================================

print(
    "Orders without product name:",
    df_sales
    .filter(F.col("product").isNull())
    .count()
)

print(
    "Orders without category:",
    df_sales
    .filter(F.col("category").isNull())
    .count()
)

Orders without product name: 0
Orders without category: 0


In [0]:
# ============================================================
# STEP 15.11 — Check Products With Missing Categories
# ============================================================

display(
    df_dim_products
    .filter(F.col("category").isNull())
    .select(
        "product_code",
        "product",
        "category"
    )
)
# ============================================================
# STEP 15.12 — Count Missing Product Categories
# ============================================================

print(
    "Products with NULL category:",
    df_dim_products
    .filter(F.col("category").isNull())
    .count()
)

product_code,product,category


Products with NULL category: 0


In [0]:
# ============================================================
# STEP 15.13 — Rebuild Sales Analytics Cleanly
# ============================================================

# Start fresh from Parent Gold Orders
df_sales = (
    spark.table("fmcg.gold.fact_orders")
    .withColumn("year", F.year("date"))
)

# Load fresh Gold dimensions
df_dim_products = spark.table("fmcg.gold.dim_products")
df_dim_gross_price = spark.table("fmcg.gold.dim_gross_price")
df_dim_customers = spark.table("fmcg.gold.dim_customers")

# -----------------------------
# Join Gross Price
# -----------------------------

df_sales = (
    df_sales.alias("o")
    .join(
        df_dim_gross_price.alias("gp"),
        (F.col("o.product_code") == F.col("gp.product_code")) &
        (F.col("o.year") == F.col("gp.year")),
        "left"
    )
    .select(
        F.col("o.date"),
        F.col("o.year"),
        F.col("o.product_code"),
        F.col("o.customer_code"),
        F.col("o.sold_quantity"),
        F.col("gp.price_inr")
    )
)

# -----------------------------
# Calculate Revenue
# -----------------------------

df_sales = (
    df_sales
    .withColumn(
        "revenue_inr",
        F.col("sold_quantity") * F.col("price_inr")
    )
)

# -----------------------------
# Join Product Dimension
# -----------------------------

df_products_lookup = (
    df_dim_products
    .select(
        F.col("product_code").alias("lookup_product_code"),
        F.col("product").alias("product_name"),
        F.col("category")
    )
)

df_sales = (
    df_sales
    .join(
        df_products_lookup,
        df_sales["product_code"] == df_products_lookup["lookup_product_code"],
        "left"
    )
    .drop("lookup_product_code")
)

display(df_sales.limit(20))

date,year,product_code,customer_code,sold_quantity,price_inr,revenue_inr,product_name,category
2024-01-01,2024,ARCHDDE20D,70002017,161,2750,442750,PX Carbon Arrow Set,Arrows
2024-01-01,2024,ARCH158F41,70002017,133,5740,763420,LX Leather Finger Tab,Finger Tab
2024-01-01,2024,ARCHAFF0E4,70002017,76,4610,350360,AX Precision Recurve Bow,Bow
2024-01-01,2024,ARCH6B94F7,70002017,92,3910,359720,AX Precision Recurve Bow,Bow
2024-01-01,2024,ARCH5D1FE7,70002017,117,1610,188370,BX Bow Stringing Tool,Bow Stringer
2024-01-01,2024,ARCH7B49A9,70002017,36,1610,57960,NX Archery Arm Guard,Arm Guard
2024-01-01,2024,ARCH497D34,70002017,98,1100,107800,TX Foam Archery Target,Target
2024-01-01,2024,ARCHE71D79,70002017,156,5830,909480,NX Archery Arm Guard,Arm Guard
2024-01-01,2024,BADM88C384,70002017,28,790,22120,AX PowerDrive Badminton Racket,Badminton Racket
2024-01-01,2024,BADMA5EBA3,70002017,33,5910,195030,BX AeroLite Badminton Racket,Badminton Racket


In [0]:
# ============================================================
# STEP 15.14 — Validate Sales Analytics
# ============================================================

print(
    "Orders without product name:",
    df_sales
    .filter(F.col("product_name").isNull())
    .count()
)

print(
    "Orders without category:",
    df_sales
    .filter(F.col("category").isNull())
    .count()
)

print(
    "Orders without price:",
    df_sales
    .filter(F.col("price_inr").isNull())
    .count()
)

print(
    "Orders without revenue:",
    df_sales
    .filter(F.col("revenue_inr").isNull())
    .count()
)

Orders without product name: 0
Orders without category: 0
Orders without price: 0
Orders without revenue: 0


In [0]:
# ============================================================
# STEP 15.15 — Inspect Customer Dimension
# ============================================================

df_dim_customers.printSchema()
display(
    df_dim_customers.limit(10)
)

root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
# ============================================================
# STEP 15.16 — Add Customer Attributes
# ============================================================

df_customers_lookup = (
    df_dim_customers
    .select(
        F.col("customer_code").alias("lookup_customer_code"),
        "customer",
        "market",
        "platform",
        "channel"
    )
)

df_sales = (
    df_sales
    .join(
        df_customers_lookup,
        df_sales["customer_code"] == df_customers_lookup["lookup_customer_code"],
        "left"
    )
    .drop("lookup_customer_code")
)

display(df_sales.limit(20))

date,year,product_code,customer_code,sold_quantity,price_inr,revenue_inr,product_name,category,customer,market,platform,channel
2024-01-01,2024,ARCHDDE20D,70002017,161,2750,442750,PX Carbon Arrow Set,Arrows,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCH158F41,70002017,133,5740,763420,LX Leather Finger Tab,Finger Tab,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCHAFF0E4,70002017,76,4610,350360,AX Precision Recurve Bow,Bow,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCH6B94F7,70002017,92,3910,359720,AX Precision Recurve Bow,Bow,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCH5D1FE7,70002017,117,1610,188370,BX Bow Stringing Tool,Bow Stringer,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCH7B49A9,70002017,36,1610,57960,NX Archery Arm Guard,Arm Guard,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCH497D34,70002017,98,1100,107800,TX Foam Archery Target,Target,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,ARCHE71D79,70002017,156,5830,909480,NX Archery Arm Guard,Arm Guard,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,BADM88C384,70002017,28,790,22120,AX PowerDrive Badminton Racket,Badminton Racket,Sporty Essentials,India,Brick & Mortar,Retailer
2024-01-01,2024,BADMA5EBA3,70002017,33,5910,195030,BX AeroLite Badminton Racket,Badminton Racket,Sporty Essentials,India,Brick & Mortar,Retailer


In [0]:
# ============================================================
# STEP 15.17 — Validate Customer Mapping
# ============================================================

print(
    "Orders without customer:",
    df_sales
    .filter(F.col("customer").isNull())
    .count()
)

print(
    "Orders without market:",
    df_sales
    .filter(F.col("market").isNull())
    .count()
)

print(
    "Orders without platform:",
    df_sales
    .filter(F.col("platform").isNull())
    .count()
)

print(
    "Orders without channel:",
    df_sales
    .filter(F.col("channel").isNull())
    .count()
)

Orders without customer: 3164
Orders without market: 3164
Orders without platform: 3164
Orders without channel: 3164


In [0]:
# ============================================================
# STEP 15.18 — Final Business Sales Dataset
# ============================================================

df_sales_final = df_sales.select(
    "date",
    "year",
    "customer_code",
    "customer",
    "market",
    "platform",
    "channel",
    "product_code",
    "product_name",
    "category",
    "sold_quantity",
    "price_inr",
    "revenue_inr"
)

df_sales_final.printSchema()

display(
    df_sales_final.limit(20)
)

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sold_quantity: long (nullable = true)
 |-- price_inr: long (nullable = true)
 |-- revenue_inr: long (nullable = true)



date,year,customer_code,customer,market,platform,channel,product_code,product_name,category,sold_quantity,price_inr,revenue_inr
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHDDE20D,PX Carbon Arrow Set,Arrows,161,2750,442750
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH158F41,LX Leather Finger Tab,Finger Tab,133,5740,763420
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHAFF0E4,AX Precision Recurve Bow,Bow,76,4610,350360
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH6B94F7,AX Precision Recurve Bow,Bow,92,3910,359720
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH5D1FE7,BX Bow Stringing Tool,Bow Stringer,117,1610,188370
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH7B49A9,NX Archery Arm Guard,Arm Guard,36,1610,57960
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH497D34,TX Foam Archery Target,Target,98,1100,107800
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHE71D79,NX Archery Arm Guard,Arm Guard,156,5830,909480
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADM88C384,AX PowerDrive Badminton Racket,Badminton Racket,28,790,22120
2024-01-01,2024,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADMA5EBA3,BX AeroLite Badminton Racket,Badminton Racket,33,5910,195030


Overall Sales KPIs


In [0]:
# ============================================================
# STEP 16.1 — Overall Sales KPIs
# ============================================================

df_sales_final.agg(
    F.sum("sold_quantity").alias("total_quantity_sold"),
    F.sum("revenue_inr").alias("total_revenue_inr"),
    F.countDistinct("customer_code").alias("unique_customers"),
    F.countDistinct("product_code").alias("unique_products"),
    F.countDistinct("date").alias("sales_days")
).show()

+-------------------+-----------------+----------------+---------------+----------+
|total_quantity_sold|total_revenue_inr|unique_customers|unique_products|sales_days|
+-------------------+-----------------+----------------+---------------+----------+
|           34561222|     105367756383|              56|            406|       169|
+-------------------+-----------------+----------------+---------------+----------+



In [0]:
# ============================================================
# STEP 16.2 — Revenue by Year
# ============================================================

display(
    df_sales_final
    .groupBy("year")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers"),
        F.countDistinct("product_code").alias("products")
    )
    .orderBy("year")
)

year,total_quantity,total_revenue,customers,products
2024,8595336,35070991330,18,347
2025,25965886,70296765053,56,393


In [0]:
# ============================================================
# STEP 16.3 — Category Performance
# ============================================================

display(
    df_sales_final
    .groupBy("category")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("product_code").alias("products"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

category,total_quantity,total_revenue,products,customers
Football,3026117,10345489260,28,18
Cricket Batting Gloves,1468352,7982130900,10,18
Dumbbells,774240,7621833220,4,18
Cricket Pads,1396006,6212228820,10,18
Football Shoes,946535,5525410970,9,18
Cricket Ball,1046961,5320029340,10,18
Cricket Kit Bag,1190771,5124440500,10,18
Cricket Helmet,874927,4949226220,9,18
Cricket Bat,1079254,4546333630,10,18
Weight Bars,467825,4145549530,2,18


In [0]:
# ============================================================
# STEP 16.4 — Top 10 Products
# ============================================================

display(
    df_sales_final
    .groupBy(
        "product_code",
        "product_name",
        "category"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

product_code,product_name,category,total_quantity,total_revenue
WLFTB7DF2B,WL Steel Barbell,Weight Bars,293042,3697666040
WLFT7AD84A,WL Hex Dumbbell,Dumbbells,209111,3297536070
WLFT053057,WL Hex Dumbbell,Dumbbells,145552,2192933600
WLFT52B261,WL Hex Dumbbell,Dumbbells,278434,1914774390
CRIC35B3C7,PX Grip Cricket Batting Gloves,Cricket Batting Gloves,169931,1502841190
CRIC538334,KX Blaze Kashmir Willow Cricket Bat,Cricket Bat,184597,1412230160
CRIC8DF8D8,AX Guard Cricket Helmet,Cricket Helmet,146937,1398319300
CRICBDB8DA,TRX Max Cricket Kit Bag,Cricket Kit Bag,187876,1363891000
FOOT7DA9E9,RX Sprint Football Boots,Football Shoes,149766,1331527340
FOOTF63A17,RX Sprint Football Boots,Football Shoes,160026,1312814160


In [0]:
# ============================================================
# STEP 16.5 — Top Customers
# ============================================================

display(
    df_sales_final
    .groupBy(
        "customer_code",
        "customer",
        "market"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

customer_code,customer,market,total_quantity,total_revenue
90002008,FitnessWorld,India,2247102,9532997850
90002009,FastTrack Sports,India,1936302,8205305690
90002016,Fitness Mania,India,1412925,5960288480
90002013,Active Gear,India,1362660,5874425360
90002005,Atlikon Official,India,1352465,5795650270
90002001,SportsHub,India,1345016,5728477020
70002017,Sporty Essentials,India,1326843,5588030530
90002004,Atlikon Sports India,India,1294410,5548912140
90002012,Ultimate Sports,India,1272267,5443598040
70002018,PlayMore Sports,India,1271831,5407843120


ADVANCED BUSINESS ANALYTICS

In [0]:
# ============================================================
# STEP 16.6 — Monthly Revenue Trend
# ============================================================

df_monthly_sales = (
    df_sales_final
    .withColumn(
        "month",
        F.date_format("date", "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy("month")
)

display(df_monthly_sales)


month,total_quantity,total_revenue,customers
2024-01,461181,1809639530,14
2024-02,477125,1892134980,14
2024-03,61202,241894770,13
2024-04,225753,893200700,13
2024-05,236201,920092800,14
2024-06,482055,1901503770,14
2024-07,446871,1760801060,13
2024-08,434278,1714438340,13
2024-09,1013218,4201544570,14
2024-10,1374279,5732458060,14


In [0]:
# ============================================================
# STEP 16.7 — Channel Performance
# ============================================================

display(
    df_sales_final
    .groupBy("channel")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

channel,total_quantity,total_revenue,customers
Retailer,19461785,82677359800,14
Direct,4976135,21307439590,4
Acquisition,9103680,1243789381,35
null,1019622,139167612,3


In [0]:
# ============================================================
# STEP 16.8 — Platform Performance
# ============================================================

display(
    df_sales_final
    .groupBy("platform")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

platform,total_quantity,total_revenue,customers
Brick & Mortar,15590326,66310121900,12
E-Commerce,8847594,37674677490,6
Sports Bar,9103680,1243789381,35
null,1019622,139167612,3


In [0]:
# ============================================================
# STEP 16.9 — Market Performance
# ============================================================

display(
    df_sales_final
    .groupBy("market")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

market,total_quantity,total_revenue,customers
India,33541600,105228588771,53
null,1019622,139167612,3


In [0]:
# ============================================================
# STEP 16.10 — Average Selling Price
# ============================================================

display(
    df_sales_final
    .groupBy("category")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue")
    )
    .withColumn(
        "average_price",
        F.col("total_revenue") / F.col("total_quantity")
    )
    .orderBy(F.desc("total_revenue"))
)

category,total_quantity,total_revenue,average_price
Football,3026117,10345489260,3418.7340608443096
Cricket Batting Gloves,1468352,7982130900,5436.115386501329
Dumbbells,774240,7621833220,9844.277252531514
Cricket Pads,1396006,6212228820,4450.0015186181145
Football Shoes,946535,5525410970,5837.513636579735
Cricket Ball,1046961,5320029340,5081.40163769233
Cricket Kit Bag,1190771,5124440500,4303.464310098248
Cricket Helmet,874927,4949226220,5656.730470084933
Cricket Bat,1079254,4546333630,4212.477906035095
Weight Bars,467825,4145549530,8861.325346016138


In [0]:
# ============================================================
# STEP 16.11 — Monthly Category Performance
# ============================================================

display(
    df_sales_final
    .withColumn(
        "month",
        F.date_format("date", "yyyy-MM")
    )
    .groupBy(
        "month",
        "category"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue")
    )
    .orderBy(
        "month",
        F.desc("total_revenue")
    )
)

month,category,total_quantity,total_revenue
2024-01,Cricket Batting Gloves,40216,213533690
2024-01,Football,44795,152150600
2024-01,Cricket Pads,31814,140435200
2024-01,Weight Bars,17071,139962510
2024-01,Cricket Ball,26249,135289350
2024-01,Football Shoes,24713,126336470
2024-01,Cricket Bat,20355,96932570
2024-01,Dumbbells,10674,71729280
2024-01,Cricket Kit Bag,19570,64644230
2024-01,Badminton Racket,17089,49505260


BUSINESS TABLES

In [0]:
# ============================================================
# STEP 17.1 — Save Monthly Sales Analytics
# ============================================================

(
    df_monthly_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_monthly_sales")
)

print("analytics_monthly_sales created.")
display(
    spark.table("fmcg.gold.analytics_monthly_sales")
)

analytics_monthly_sales created.


month,total_quantity,total_revenue,customers
2024-01,461181,1809639530,14
2024-02,477125,1892134980,14
2024-03,61202,241894770,13
2024-04,225753,893200700,13
2024-05,236201,920092800,14
2024-06,482055,1901503770,14
2024-07,446871,1760801060,13
2024-08,434278,1714438340,13
2024-09,1013218,4201544570,14
2024-10,1374279,5732458060,14


In [0]:
# ============================================================
# STEP 17.2 — Save Category Sales Analytics
# ============================================================

df_category_sales = (
    df_sales_final
    .groupBy("category")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("product_code").alias("products"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    df_category_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_category_sales")
)

print("analytics_category_sales created.")

analytics_category_sales created.


In [0]:
# ============================================================
# STEP 17.2 — Save Category Sales Analytics
# ============================================================

df_category_sales = (
    df_sales_final
    .groupBy("category")
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("product_code").alias("products"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    df_category_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_category_sales")
)

print("analytics_category_sales created.")

analytics_category_sales created.


In [0]:
# ============================================================
# STEP 17.3 — Save Product Sales Analytics
# ============================================================

df_product_sales = (
    df_sales_final
    .groupBy(
        "product_code",
        "product_name",
        "category"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    df_product_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_product_sales")
)

print("analytics_product_sales created.")

analytics_product_sales created.


In [0]:
# ============================================================
# STEP 17.3 — Save Product Sales Analytics
# ============================================================

df_product_sales = (
    df_sales_final
    .groupBy(
        "product_code",
        "product_name",
        "category"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    df_product_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_product_sales")
)

print("analytics_product_sales created.")

analytics_product_sales created.


In [0]:
# ============================================================
# STEP 17.4 — Save Customer Sales Analytics
# ============================================================

df_customer_sales = (
    df_sales_final
    .groupBy(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel"
    )
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("product_code").alias("products_purchased")
    )
    .orderBy(F.desc("total_revenue"))
)

(
    df_customer_sales.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("fmcg.gold.analytics_customer_sales")
)

print("analytics_customer_sales created.")

analytics_customer_sales created.


In [0]:
# ============================================================
# STEP 18.1 — Final Gold Tables
# ============================================================

display(
    spark.sql("SHOW TABLES IN fmcg.gold")
)

database,tableName,isTemporary
gold,analytics_category_sales,false
gold,analytics_customer_sales,false
gold,analytics_monthly_sales,false
gold,analytics_product_sales,false
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,sb_dim_customers,false


In [0]:
# ============================================================
# STEP 18.2 — Analytics Row Counts
# ============================================================

analytics_tables = [
    "analytics_monthly_sales",
    "analytics_category_sales",
    "analytics_product_sales",
    "analytics_customer_sales"
]

for table_name in analytics_tables:
    count = spark.table(f"fmcg.gold.{table_name}").count()
    print(f"{table_name}: {count:,} rows")

analytics_monthly_sales: 23 rows
analytics_category_sales: 117 rows
analytics_product_sales: 406 rows
analytics_customer_sales: 56 rows


In [0]:
# ============================================================
# STEP 18.3 — Final Pipeline Health Check
# ============================================================

checks = {
    "Bronze Customers": spark.table("fmcg.bronze.customers").count(),
    "Bronze Products": spark.table("fmcg.bronze.products").count(),
    "Bronze Gross Price": spark.table("fmcg.bronze.gross_price").count(),
    "Bronze Orders": spark.table("fmcg.bronze.orders").count(),

    "Silver Customers": spark.table("fmcg.silver.customers").count(),
    "Silver Products": spark.table("fmcg.silver.products").count(),
    "Silver Gross Price": spark.table("fmcg.silver.gross_price").count(),
    "Silver Orders": spark.table("fmcg.silver.orders").count(),

    "Gold Customers": spark.table("fmcg.gold.dim_customers").count(),
    "Gold Products": spark.table("fmcg.gold.dim_products").count(),
    "Gold Gross Price": spark.table("fmcg.gold.dim_gross_price").count(),
    "Gold Orders": spark.table("fmcg.gold.fact_orders").count(),

    "Analytics Monthly": spark.table("fmcg.gold.analytics_monthly_sales").count(),
    "Analytics Category": spark.table("fmcg.gold.analytics_category_sales").count(),
    "Analytics Product": spark.table("fmcg.gold.analytics_product_sales").count(),
    "Analytics Customer": spark.table("fmcg.gold.analytics_customer_sales").count()
}

for name, count in checks.items():
    print(f"{name:<25} : {count:,}")

Bronze Customers          : 39
Bronze Products           : 20
Bronze Gross Price        : 108
Bronze Orders             : 51,810
Silver Customers          : 35
Silver Products           : 17
Silver Gross Price        : 75
Silver Orders             : 51,810
Gold Customers            : 53
Gold Products             : 414
Gold Gross Price          : 811
Gold Orders               : 128,180
Analytics Monthly         : 23
Analytics Category        : 117
Analytics Product         : 406
Analytics Customer        : 56


In [0]:
# ============================================================
# STEP 18.4 — Final Revenue Check
# ============================================================

df_final_kpi = (
    df_sales_final
    .agg(
        F.sum("sold_quantity").alias("total_quantity"),
        F.sum("revenue_inr").alias("total_revenue"),
        F.countDistinct("customer_code").alias("customers"),
        F.countDistinct("product_code").alias("products"),
        F.min("date").alias("first_order_date"),
        F.max("date").alias("last_order_date")
    )
)

display(df_final_kpi)

total_quantity,total_revenue,customers,products,first_order_date,last_order_date
34561222,105367756383,56,406,2024-01-01,2025-11-30


Create a Test Incremental File


In [0]:
# ============================================================
# STEP 19.1 — Create Test Incremental Orders
# ============================================================

from pyspark.sql import functions as F

df_incremental_test = (
    df_orders_raw
    .limit(100)
    .withColumn(
        "order_id",
        F.concat(
            F.col("order_id"),
            F.lit("_INC001")
        )
    )
)

print("Incremental test records:", df_incremental_test.count())

display(df_incremental_test.limit(10))

Incremental test records: 100


order_id,order_placement_date,customer_id,product_id,order_qty
FOCT62720602_INC001,"Tuesday, September 30, 2025",ABC987,25891301,71.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891502,125.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891403,462.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",INVALID,25891601,133.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891602,79.0
FOCT62720602_INC001,2025/09/30,XYZ123,25891101,472.0
FOCT62720602_INC001,2025/09/30,ABC987,25891301,71.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891303,24.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891301,26.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891102,328.0


In [0]:
# ============================================================
# STEP 19.2 — Write Incremental File to S3
# ============================================================

(
    df_incremental_test
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("s3://sportsbar-dp24124/orders/incremental/")
)

print("Incremental test file written to S3.")

Incremental test file written to S3.


In [0]:
# ============================================================
# STEP 19.3 — Read Incremental Orders
# ============================================================

df_orders_incremental = spark.read.csv(
    "s3://sportsbar-dp24124/orders/incremental/",
    header=True,
    inferSchema=True
)

print(
    "Incremental rows:",
    df_orders_incremental.count()
)

df_orders_incremental.printSchema()

display(df_orders_incremental.limit(10))

Incremental rows: 100
root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)



order_id,order_placement_date,customer_id,product_id,order_qty
FOCT62720602_INC001,"Tuesday, September 30, 2025",ABC987,25891301,71.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891502,125.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891403,462.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",INVALID,25891601,133.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891602,79.0
FOCT62720602_INC001,2025/09/30,XYZ123,25891101,472.0
FOCT62720602_INC001,2025/09/30,ABC987,25891301,71.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891303,24.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891301,26.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891102,328.0


In [0]:
# ============================================================
# STEP 19.4 — Prepare Incremental Bronze Data
# ============================================================

df_incremental_bronze = (
    df_orders_incremental
    .withColumn(
        "read_timestamp",
        F.current_timestamp()
    )
    .select(
        "*",
        "_metadata.file_name",
        "_metadata.file_size"
    )
)

df_incremental_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
# ============================================================
# STEP 19.5 — Identify New Orders
# ============================================================

df_existing_orders = (
    spark.table("fmcg.bronze.orders")
    .select("order_id")
    .distinct()
)

df_new_orders = (
    df_incremental_bronze
    .join(
        df_existing_orders,
        on="order_id",
        how="left_anti"
    )
)

print(
    "New orders:",
    df_new_orders.count()
)

New orders: 100


In [0]:
# ============================================================
# STEP 19.6 — Duplicate Protection Check
# ============================================================

df_second_check = (
    df_new_orders
    .join(
        spark.table("fmcg.bronze.orders")
        .select("order_id")
        .distinct(),
        on="order_id",
        how="left_anti"
    )
)

print(
    "Records still new:",
    df_second_check.count()
)

Records still new: 100


In [0]:
# ============================================================
# STEP 19.7 — Append New Orders to Bronze
# ============================================================

(
    df_new_orders
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("fmcg.bronze.orders")
)

print("New orders appended to Bronze.")
print(
    "Bronze Orders:",
    spark.table("fmcg.bronze.orders").count()
)

New orders appended to Bronze.
Bronze Orders: 51910


In [0]:
# ============================================================
# STEP 19.8 — Read Incremental Orders from Bronze
# ============================================================

df_orders_incremental_bronze = (
    spark.table("fmcg.bronze.orders")
    .filter(F.col("order_id").endswith("_INC001"))
)

print(
    "Incremental Bronze orders:",
    df_orders_incremental_bronze.count()
)

display(
    df_orders_incremental_bronze.limit(10)
)

Incremental Bronze orders: 100


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FOCT62720602_INC001,"Tuesday, September 30, 2025",ABC987,25891301,71.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891502,125.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891403,462.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,"Tuesday, September 30, 2025",INVALID,25891601,133.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,"Tuesday, September 30, 2025",789720,25891602,79.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,2025/09/30,XYZ123,25891101,472.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62720602_INC001,2025/09/30,ABC987,25891301,71.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891303,24.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891301,26.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933
FOCT62402303_INC001,"Tuesday, September 30, 2025",789402,25891102,328.0,2026-09-21T18:09:00.439Z,part-00000-tid-7783462897801950988-4eabd8a2-c40c-4396-a7c7-a00a1c00cb88-1952-1-c000.csv,6933


In [0]:
# ============================================================
# STEP 19.9 — Convert Incremental Order Date
# Handles multiple source date formats
# ============================================================

df_orders_incremental_silver = (
    df_orders_incremental_bronze
    .withColumn(
        "order_date",
        F.coalesce(
            F.expr(
                "try_to_timestamp(order_placement_date, 'yyyy/MM/dd')"
            ).cast("date"),

            F.expr(
                "try_to_timestamp(order_placement_date, 'yyyy-MM-dd')"
            ).cast("date"),

            F.expr(
                "try_to_timestamp(order_placement_date, 'EEEE, MMMM d, yyyy')"
            ).cast("date")
        )
    )
)

display(
    df_orders_incremental_silver.select(
        "order_id",
        "order_placement_date",
        "order_date",
        "customer_id",
        "product_id",
        "order_qty"
    ).limit(20)
)

order_id,order_placement_date,order_date,customer_id,product_id,order_qty
FOCT62720602_INC001,"Tuesday, September 30, 2025",2025-09-30,ABC987,25891301,71.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",2025-09-30,789720,25891502,125.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",2025-09-30,789720,25891403,462.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",2025-09-30,INVALID,25891601,133.0
FOCT62720602_INC001,"Tuesday, September 30, 2025",2025-09-30,789720,25891602,79.0
FOCT62720602_INC001,2025/09/30,2025-09-30,XYZ123,25891101,472.0
FOCT62720602_INC001,2025/09/30,2025-09-30,ABC987,25891301,71.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",2025-09-30,789402,25891303,24.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",2025-09-30,789402,25891301,26.0
FOCT62402303_INC001,"Tuesday, September 30, 2025",2025-09-30,789402,25891102,328.0


In [0]:
# ============================================================
# STEP 19.10 — Validate Incremental Order Dates
# ============================================================

print(
    "Invalid incremental dates:",
    df_orders_incremental_silver
    .filter(F.col("order_date").isNull())
    .count()
)

print(
    "Valid incremental dates:",
    df_orders_incremental_silver
    .filter(F.col("order_date").isNotNull())
    .count()
)

Invalid incremental dates: 8
Valid incremental dates: 92


In [0]:
# ============================================================
# STEP 19.11 — Validate Incremental Quantities
# ============================================================

print(
    "NULL quantities:",
    df_orders_incremental_silver
    .filter(F.col("order_qty").isNull())
    .count()
)

print(
    "Zero quantities:",
    df_orders_incremental_silver
    .filter(F.col("order_qty") == 0)
    .count()
)

print(
    "Negative quantities:",
    df_orders_incremental_silver
    .filter(F.col("order_qty") < 0)
    .count()
)

NULL quantities: 2
Zero quantities: 0
Negative quantities: 0


In [0]:
# ============================================================
# STEP 19.12 — Separate Invalid Incremental Orders
# ============================================================

df_incremental_rejected = (
    df_orders_incremental_silver
    .filter(
        F.col("order_qty").isNull()
        | (F.col("order_qty") <= 0)
    )
)

df_incremental_valid = (
    df_orders_incremental_silver
    .filter(
        F.col("order_qty").isNotNull()
        & (F.col("order_qty") > 0)
    )
)

print(
    "Rejected incremental orders:",
    df_incremental_rejected.count()
)

print(
    "Valid incremental orders:",
    df_incremental_valid.count()
)

Rejected incremental orders: 2
Valid incremental orders: 98


In [0]:
# ============================================================
# STEP 19.13 — Inspect Rejected Incremental Orders
# ============================================================

display(
    df_incremental_rejected.select(
        "order_id",
        "order_placement_date",
        "order_date",
        "customer_id",
        "product_id",
        "order_qty"
    )
)

order_id,order_placement_date,order_date,customer_id,product_id,order_qty
FOCT63221602_INC001,"Tuesday, September 30, 2025",2025-09-30,789221,25891602,null
FOCT63221602_INC001,"Tuesday, September 30, 2025",2025-09-30,789221,25891602,null


In [0]:
# ============================================================
# STEP 19.14 — Validate Clean Incremental Orders
# ============================================================

print(
    "Valid incremental orders:",
    df_incremental_valid.count()
)

print(
    "NULL order dates:",
    df_incremental_valid
    .filter(F.col("order_date").isNull())
    .count()
)

print(
    "NULL quantities:",
    df_incremental_valid
    .filter(F.col("order_qty").isNull())
    .count()
)

print(
    "Zero quantities:",
    df_incremental_valid
    .filter(F.col("order_qty") == 0)
    .count()
)

print(
    "Negative quantities:",
    df_incremental_valid
    .filter(F.col("order_qty") < 0)
    .count()
)

Valid incremental orders: 98
NULL order dates: 8
NULL quantities: 0
Zero quantities: 0
Negative quantities: 0


In [0]:
# ============================================================
# STEP 19.15 — Map Product Codes
# ============================================================

df_products_silver = spark.table("fmcg.silver.products")

df_incremental_valid = (
    df_incremental_valid.alias("o")
    .join(
        df_products_silver
        .select(
            F.col("product_id").alias("lookup_product_id"),
            F.col("product_code")
        )
        .alias("p"),
        F.col("o.product_id") == F.col("p.lookup_product_id"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("p.product_code")
    )
)

display(
    df_incremental_valid.select(
        "order_id",
        "product_id",
        "product_code",
        "order_qty"
    ).limit(20)
)

order_id,product_id,product_code,order_qty
FOCT62720602_INC001,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,71.0
FOCT62720602_INC001,25891502,null,125.0
FOCT62720602_INC001,25891403,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,462.0
FOCT62720602_INC001,25891601,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,133.0
FOCT62720602_INC001,25891602,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,79.0
FOCT62720602_INC001,25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,472.0
FOCT62720602_INC001,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,71.0
FOCT62402303_INC001,25891303,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,24.0
FOCT62402303_INC001,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,26.0
FOCT62402303_INC001,25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,328.0


In [0]:
# ============================================================
# STEP 19.16 — Validate Product Mapping
# ============================================================

print(
    "Incremental orders without product_code:",
    df_incremental_valid
    .filter(F.col("product_code").isNull())
    .count()
)

Incremental orders without product_code: 10


In [0]:
# ============================================================
# STEP 19.17 — Identify Unmapped Incremental Products
# ============================================================

display(
    df_incremental_valid
    .filter(F.col("product_code").isNull())
    .select(
        "product_id",
        "order_id",
        "customer_id",
        "order_qty",
        "order_date"
    )
    .groupBy("product_id")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("order_qty").alias("total_quantity")
    )
    .orderBy("product_id")
)

product_id,order_count,total_quantity
25891502,7,1109.0
77777777,2,701.0
99999999,1,406.0


In [0]:
# ============================================================
# STEP 19.18 — Check Unmapped Products Against Product Master
# ============================================================

unmapped_product_ids = (
    df_incremental_valid
    .filter(F.col("product_code").isNull())
    .select("product_id")
    .distinct()
)

display(
    unmapped_product_ids.alias("u")
    .join(
        spark.table("fmcg.bronze.products").alias("p"),
        F.col("u.product_id").cast("string") ==
        F.col("p.product_id").cast("string"),
        "left"
    )
    .select(
        F.col("u.product_id").alias("incremental_product_id"),
        F.col("p.product_id").alias("bronze_product_id"),
        F.col("p.product_name"),
        F.col("p.category")
    )
)





































































































































































































































































incremental_product_id,bronze_product_id,product_name,category
77777777,null,null,null
99999999,null,null,null
25891502,null,null,null


In [0]:
# ============================================================
# STEP 19.19 — Separate Unmapped Product Orders
# ============================================================

df_unmapped_product_orders = (
    df_incremental_valid
    .filter(F.col("product_code").isNull())
)

df_incremental_product_valid = (
    df_incremental_valid
    .filter(F.col("product_code").isNotNull())
)

print(
    "Orders with unknown products:",
    df_unmapped_product_orders.count()
)

print(
    "Orders with valid products:",
    df_incremental_product_valid.count()
)

Orders with unknown products: 10
Orders with valid products: 88


In [0]:
# ============================================================
# STEP 19.19 — Separate Unknown Product Orders
# ============================================================

df_unmapped_product_orders = (
    df_incremental_valid
    .filter(F.col("product_code").isNull())
)

df_incremental_product_valid = (
    df_incremental_valid
    .filter(F.col("product_code").isNotNull())
)

print(
    "Orders with unknown products:",
    df_unmapped_product_orders.count()
)

print(
    "Orders with valid products:",
    df_incremental_product_valid.count()
)

Orders with unknown products: 10
Orders with valid products: 88


In [0]:
# ============================================================
# STEP 19.20 — Map Customer Codes
# ============================================================

df_customers_silver = spark.table("fmcg.silver.customers")

df_incremental_product_valid = (
    df_incremental_product_valid.alias("o")
    .join(
        df_customers_silver
        .select(
            F.col("customer_id").alias("lookup_customer_id")
        )
        .alias("c"),
        F.col("o.customer_id").cast("string") ==
        F.col("c.lookup_customer_id").cast("string"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("c.lookup_customer_id").cast("string").alias("customer_code")
    )
)

display(
    df_incremental_product_valid.select(
        "order_id",
        "customer_id",
        "customer_code",
        "product_id",
        "product_code",
        "order_qty"
    ).limit(20)
)

order_id,customer_id,customer_code,product_id,product_code,order_qty
FOCT62720602_INC001,ABC987,null,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,71.0
FOCT62720602_INC001,789720,789720,25891403,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,462.0
FOCT62720602_INC001,INVALID,null,25891601,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,133.0
FOCT62720602_INC001,789720,789720,25891602,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,79.0
FOCT62720602_INC001,XYZ123,null,25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,472.0
FOCT62720602_INC001,ABC987,null,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,71.0
FOCT62402303_INC001,789402,789402,25891303,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,24.0
FOCT62402303_INC001,789402,789402,25891301,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,26.0
FOCT62402303_INC001,789402,789402,25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,328.0
FOCT62402303_INC001,789402,789402,25891103,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,454.0


In [0]:
# ============================================================
# STEP 19.21 — Validate Customer Mapping
# ============================================================

print(
    "Incremental orders without customer_code:",
    df_incremental_product_valid
    .filter(F.col("customer_code").isNull())
    .count()
)

Incremental orders without customer_code: 14


In [0]:
# ============================================================
# STEP 19.22 — Prepare Incremental Gold Records
# ============================================================

df_incremental_gold = (
    df_incremental_product_valid
    .select(
        F.col("order_date").alias("date"),
        "product_code",
        "customer_code",
        F.col("order_qty").cast("long").alias("sold_quantity")
    )
)

print(
    "Incremental Gold records:",
    df_incremental_gold.count()
)

display(df_incremental_gold.limit(20))

Incremental Gold records: 88


date,product_code,customer_code,sold_quantity
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,null,71
2025-09-30,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789720,462
2025-09-30,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,null,133
2025-09-30,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789720,79
2025-09-30,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,null,472
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,null,71
2025-09-30,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789402,24
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,789402,26
2025-09-30,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789402,328
2025-09-30,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,789402,454


In [0]:
# ============================================================
# STEP 19.23 — Aggregate Incremental Orders
# ============================================================

df_incremental_gold_agg = (
    df_incremental_gold
    .groupBy(
        "date",
        "product_code",
        "customer_code"
    )
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
)

print(
    "Aggregated incremental Gold records:",
    df_incremental_gold_agg.count()
)

display(
    df_incremental_gold_agg.limit(20)
)

Aggregated incremental Gold records: 73


date,product_code,customer_code,sold_quantity
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,null,142
2025-09-30,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789402,328
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,789102,20
2025-09-30,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789501,334
2025-09-30,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789603,87
2025-09-30,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789401,460
2025-09-30,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789720,220
2025-09-30,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,789101,204
2025-09-30,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,null,25
2025-09-30,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789622,494


In [0]:
# ============================================================
# STEP 19.24 — Validate Incremental Gold
# ============================================================

print(
    "NULL dates:",
    df_incremental_gold_agg
    .filter(F.col("date").isNull())
    .count()
)

print(
    "NULL product codes:",
    df_incremental_gold_agg
    .filter(F.col("product_code").isNull())
    .count()
)

print(
    "NULL customer codes:",
    df_incremental_gold_agg
    .filter(F.col("customer_code").isNull())
    .count()
)

print(
    "NULL quantities:",
    df_incremental_gold_agg
    .filter(F.col("sold_quantity").isNull())
    .count()
)

print(
    "Negative quantities:",
    df_incremental_gold_agg
    .filter(F.col("sold_quantity") < 0)
    .count()
)

NULL dates: 7
NULL product codes: 0
NULL customer codes: 9
NULL quantities: 0
Negative quantities: 0


In [0]:
from delta.tables import DeltaTable

In [0]:
# ============================================================
# STEP 19.25 — Incremental MERGE Into Parent Gold
# ============================================================

orders_gold_target = DeltaTable.forName(
    spark,
    "fmcg.gold.fact_orders"
)

(
    orders_gold_target.alias("target")
    .merge(
        df_incremental_gold_agg.alias("source"),
        """
        target.date = source.date
        AND target.product_code = source.product_code
        AND target.customer_code = source.customer_code
        """
    )
    .whenMatchedUpdate(
        set={
            "sold_quantity":
                "target.sold_quantity + source.sold_quantity"
        }
    )
    .whenNotMatchedInsert(
        values={
            "date": "source.date",
            "product_code": "source.product_code",
            "customer_code": "source.customer_code",
            "sold_quantity": "source.sold_quantity"
        }
    )
    .execute()
)

print("Incremental Orders MERGE completed.")

Incremental Orders MERGE completed.


In [0]:
# ============================================================
# STEP 19.26 — Verify Parent Gold
# ============================================================

df_gold_orders_after_incremental = (
    spark.table("fmcg.gold.fact_orders")
)

print(
    "Gold Orders after incremental MERGE:",
    df_gold_orders_after_incremental.count()
)

Gold Orders after incremental MERGE: 128195


In [0]:
# ============================================================
# STEP 19.27 — Verify Incremental Records in Gold
# ============================================================

df_incremental_present = (
    df_gold_orders_after_incremental.alias("g")
    .join(
        df_incremental_gold_agg.alias("i"),
        (
            (F.col("g.date") == F.col("i.date")) &
            (F.col("g.product_code") == F.col("i.product_code")) &
            (F.col("g.customer_code") == F.col("i.customer_code"))
        ),
        "inner"
    )
    .select(
        F.col("g.date"),
        F.col("g.product_code"),
        F.col("g.customer_code"),
        F.col("g.sold_quantity")
    )
    .distinct()
)

print(
    "Incremental keys present in Gold:",
    df_incremental_present.count()
)

Incremental keys present in Gold: 58


In [0]:
# ============================================================
# STEP 19.28 — Verify Gold Grain
# ============================================================

print(
    "Duplicate Gold keys:",
    df_gold_orders_after_incremental
    .groupBy(
        "date",
        "product_code",
        "customer_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate Gold keys: 0


In [0]:
# ============================================================
# STEP 19.29 — Capture Current Gold Quantities
# ============================================================

df_before_rerun = (
    df_gold_orders_after_incremental.alias("g")
    .join(
        df_incremental_gold_agg.alias("i"),
        (
            (F.col("g.date") == F.col("i.date")) &
            (F.col("g.product_code") == F.col("i.product_code")) &
            (F.col("g.customer_code") == F.col("i.customer_code"))
        ),
        "inner"
    )
    .select(
        F.col("g.date"),
        F.col("g.product_code"),
        F.col("g.customer_code"),
        F.col("g.sold_quantity").alias("quantity_before_rerun")
    )
)

print(
    "Keys before rerun:",
    df_before_rerun.count()
)

Keys before rerun: 58


In [0]:
# ============================================================
# STEP 19.30 — Re-run Incremental MERGE
# ============================================================

orders_gold_target = DeltaTable.forName(
    spark,
    "fmcg.gold.fact_orders"
)

(
    orders_gold_target.alias("target")
    .merge(
        df_incremental_gold_agg.alias("source"),
        """
        target.date = source.date
        AND target.product_code = source.product_code
        AND target.customer_code = source.customer_code
        """
    )
    .whenMatchedUpdate(
        set={
            "sold_quantity":
                "target.sold_quantity + source.sold_quantity"
        }
    )
    .whenNotMatchedInsert(
        values={
            "date": "source.date",
            "product_code": "source.product_code",
            "customer_code": "source.customer_code",
            "sold_quantity": "source.sold_quantity"
        }
    )
    .execute()
)

print("Second MERGE completed.")

Second MERGE completed.


In [0]:
# ============================================================
# STEP 19.31 — Check Effect of Second MERGE
# ============================================================

df_after_rerun = (
    spark.table("fmcg.gold.fact_orders").alias("g")
    .join(
        df_incremental_gold_agg.alias("i"),
        (
            (F.col("g.date") == F.col("i.date")) &
            (F.col("g.product_code") == F.col("i.product_code")) &
            (F.col("g.customer_code") == F.col("i.customer_code"))
        ),
        "inner"
    )
    .select(
        F.col("g.date"),
        F.col("g.product_code"),
        F.col("g.customer_code"),
        F.col("g.sold_quantity").alias("quantity_after_rerun")
    )
)

display(
    df_before_rerun.alias("b")
    .join(
        df_after_rerun.alias("a"),
        ["date", "product_code", "customer_code"],
        "inner"
    )
    .withColumn(
        "quantity_change",
        F.col("a.quantity_after_rerun") -
        F.col("b.quantity_before_rerun")
    )
    .limit(20)
)

date,product_code,customer_code,quantity_before_rerun,quantity_after_rerun,quantity_change
2025-09-30,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789402,984,984,0
2025-09-30,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,789102,60,60,0
2025-09-30,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789501,1002,1002,0
2025-09-30,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789603,261,261,0
2025-09-30,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789401,1380,1380,0
2025-09-30,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789720,660,660,0
2025-09-30,91a67435e55806eae89d739f013ea158a75b7affc758b9116d43937b691ee4a9,789101,612,612,0
2025-09-30,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,789622,1482,1482,0
2025-09-30,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789903,1938,1938,0
2025-09-30,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,789501,294,294,0


In [0]:
# ============================================================
# STEP 19.32.1 — Create Incremental Batch Audit Table
# ============================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS fmcg.gold.pipeline_batch_audit (
    pipeline_name STRING,
    batch_id STRING,
    source_file STRING,
    records_processed BIGINT,
    records_rejected BIGINT,
    processed_timestamp TIMESTAMP,
    status STRING
)
USING DELTA
""")

print("Pipeline batch audit table ready.")

Pipeline batch audit table ready.


In [0]:
# ============================================================
# STEP 19.32.2 — Define Incremental Batch
# ============================================================

batch_id = "ORDERS_INC001"
pipeline_name = "fmcg_orders_incremental"
source_file = "orders_incremental_INC001"

print("Batch ID:", batch_id)
print("Pipeline:", pipeline_name)

Batch ID: ORDERS_INC001
Pipeline: fmcg_orders_incremental


In [0]:
# ============================================================
# STEP 19.32.3 — Check Batch Processing Status
# ============================================================

batch_already_processed = (
    spark.table("fmcg.gold.pipeline_batch_audit")
    .filter(
        (F.col("pipeline_name") == pipeline_name) &
        (F.col("batch_id") == batch_id) &
        (F.col("status") == "SUCCESS")
    )
    .count()
)

print(
    "Batch already processed:",
    batch_already_processed
)

Batch already processed: 0


In [0]:
# ============================================================
# STEP 19.32.4 — Record Successful Batch
# ============================================================

audit_record = spark.createDataFrame(
    [
        (
            pipeline_name,
            batch_id,
            source_file,
            df_incremental_product_valid.count(),
            df_incremental_valid.filter(
                F.col("product_code").isNull()
            ).count(),
            "SUCCESS"
        )
    ],
    [
        "pipeline_name",
        "batch_id",
        "source_file",
        "records_processed",
        "records_rejected",
        "status"
    ]
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

(
    audit_record
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("fmcg.gold.pipeline_batch_audit")
)

print("Batch recorded successfully.")

Batch recorded successfully.


In [0]:
# ============================================================
# STEP 19.32.5 — Verify Batch Audit
# ============================================================

display(
    spark.table("fmcg.gold.pipeline_batch_audit")
    .filter(
        (F.col("pipeline_name") == pipeline_name) &
        (F.col("batch_id") == batch_id)
    )
)

pipeline_name,batch_id,source_file,records_processed,records_rejected,processed_timestamp,status
fmcg_orders_incremental,ORDERS_INC001,orders_incremental_INC001,88,10,2026-09-21T18:39:47.996Z,SUCCESS


In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY fmcg.gold.fact_orders
    """)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .orderBy(F.desc("version"))
    .limit(10)
)

version,timestamp,operation,operationMetrics
4,2026-09-21T18:35:06.000Z,OPTIMIZE,"Map(numRemovedFiles -> 4, numRemovedBytes -> 241248, p25FileSize -> 250690, numDeletionVectorsRemoved -> 2, minFileSize -> 250690, numAddedFiles -> 1, maxFileSize -> 250690, p75FileSize -> 250690, p50FileSize -> 250690, numAddedBytes -> 250690)"
3,2026-09-21T18:35:04.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2755, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 58, executionTimeMs -> 4449, materializeSourceTimeMs -> 725, numTargetRowsInserted -> 15, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1733, numTargetRowsUpdated -> 58, numOutputRows -> 73, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 73, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1879)"
2,2026-09-21T18:32:08.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2755, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 58, executionTimeMs -> 5174, materializeSourceTimeMs -> 900, numTargetRowsInserted -> 15, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1873, numTargetRowsUpdated -> 58, numOutputRows -> 73, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 73, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2342)"
1,2026-09-21T17:07:00.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 102684, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3663, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 35125, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2085, numTargetRowsUpdated -> 0, numOutputRows -> 35125, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 35125, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1483)"
0,2026-09-21T10:08:31.000Z,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 93055, numOutputBytes -> 133054)"


POWER BI DASHBOARD


In [0]:
# ============================================================
# POWER BI STEP 1 — Verify Analytics Tables
# ============================================================

display(
    spark.sql("""
        SHOW TABLES IN fmcg.gold
    """)
)

database,tableName,isTemporary
gold,analytics_category_sales,false
gold,analytics_customer_sales,false
gold,analytics_monthly_sales,false
gold,analytics_product_sales,false
gold,dim_customers,false
gold,dim_date,false
gold,dim_gross_price,false
gold,dim_products,false
gold,fact_orders,false
gold,pipeline_batch_audit,false


In [0]:
# ============================================================
# POWER BI STEP 2 — Inspect Analytics Schemas
# ============================================================

for table_name in [
    "analytics_monthly_sales",
    "analytics_category_sales",
    "analytics_product_sales",
    "analytics_customer_sales"
]:
    
    print("\n========================================")
    print(table_name)
    print("========================================")
    
    spark.table(
        f"fmcg.gold.{table_name}"
    ).printSchema()


analytics_monthly_sales
root
 |-- month: string (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_revenue: long (nullable = true)
 |-- customers: long (nullable = true)


analytics_category_sales
root
 |-- category: string (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_revenue: long (nullable = true)
 |-- products: long (nullable = true)
 |-- customers: long (nullable = true)


analytics_product_sales
root
 |-- product_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_revenue: long (nullable = true)
 |-- customers: long (nullable = true)


analytics_customer_sales
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- tot

In [0]:
# ============================================================
# POWER BI STEP 3 — Preview Analytics Tables
# ============================================================

display(
    spark.table("fmcg.gold.analytics_monthly_sales")
)

display(
    spark.table("fmcg.gold.analytics_category_sales")
)

display(
    spark.table("fmcg.gold.analytics_product_sales")
    .limit(20)
)

display(
    spark.table("fmcg.gold.analytics_customer_sales")
    .limit(20)
)

month,total_quantity,total_revenue,customers
2024-01,461181,1809639530,14
2024-02,477125,1892134980,14
2024-03,61202,241894770,13
2024-04,225753,893200700,13
2024-05,236201,920092800,14
2024-06,482055,1901503770,14
2024-07,446871,1760801060,13
2024-08,434278,1714438340,13
2024-09,1013218,4201544570,14
2024-10,1374279,5732458060,14


category,total_quantity,total_revenue,products,customers
Football,3026117,10345489260,28,18
Cricket Batting Gloves,1468352,7982130900,10,18
Dumbbells,774240,7621833220,4,18
Cricket Pads,1396006,6212228820,10,18
Football Shoes,946535,5525410970,9,18
Cricket Ball,1046961,5320029340,10,18
Cricket Kit Bag,1190771,5124440500,10,18
Cricket Helmet,874927,4949226220,9,18
Cricket Bat,1079254,4546333630,10,18
Weight Bars,467825,4145549530,2,18


product_code,product_name,category,total_quantity,total_revenue,customers
WLFTB7DF2B,WL Steel Barbell,Weight Bars,293042,3697666040,18
WLFT7AD84A,WL Hex Dumbbell,Dumbbells,209111,3297536070,18
WLFT053057,WL Hex Dumbbell,Dumbbells,145552,2192933600,18
WLFT52B261,WL Hex Dumbbell,Dumbbells,278434,1914774390,18
CRIC35B3C7,PX Grip Cricket Batting Gloves,Cricket Batting Gloves,169931,1502841190,18
CRIC538334,KX Blaze Kashmir Willow Cricket Bat,Cricket Bat,184597,1412230160,18
CRIC8DF8D8,AX Guard Cricket Helmet,Cricket Helmet,146937,1398319300,18
CRICBDB8DA,TRX Max Cricket Kit Bag,Cricket Kit Bag,187876,1363891000,18
FOOT7DA9E9,RX Sprint Football Boots,Football Shoes,149766,1331527340,18
FOOTF63A17,RX Sprint Football Boots,Football Shoes,160026,1312814160,18


customer_code,customer,market,platform,channel,total_quantity,total_revenue,products_purchased
90002008,FitnessWorld,India,E-Commerce,Retailer,2247102,9532997850,389
90002009,FastTrack Sports,India,Brick & Mortar,Retailer,1936302,8205305690,389
90002016,Fitness Mania,India,Brick & Mortar,Retailer,1412925,5960288480,389
90002013,Active Gear,India,E-Commerce,Retailer,1362660,5874425360,389
90002005,Atlikon Official,India,E-Commerce,Direct,1352465,5795650270,389
90002001,SportsHub,India,E-Commerce,Retailer,1345016,5728477020,389
70002017,Sporty Essentials,India,Brick & Mortar,Retailer,1326843,5588030530,389
90002004,Atlikon Sports India,India,Brick & Mortar,Direct,1294410,5548912140,389
90002012,Ultimate Sports,India,E-Commerce,Retailer,1272267,5443598040,389
70002018,PlayMore Sports,India,Brick & Mortar,Retailer,1271831,5407843120,389
